In [ ]:
#REDACTED_GOOGLE_API_KEY
#REDACTED_OPENAI_KEY

In [1]:
!pip install google-api-python-client
!pip install youtube-transcript-api openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 19.3 MB/s eta 0:00:00


In [2]:
from googleapiclient.discovery import build
import pandas as pd

# 🔑 Replace with your API key
API_KEY = "REDACTED_GOOGLE_API_KEY"

youtube = build("youtube", "v3", developerKey=API_KEY)

QUERY = "Full Python Course"
MAX_VIDEOS = 20
MAX_COMMENTS = 100

videos_data = []
comments_data = []

# 🔍 Search videos
search_response = youtube.search().list(
    q=QUERY,
    part="id,snippet",
    type="video",
    maxResults=MAX_VIDEOS
).execute()

for item in search_response["items"]:
    video_id = item["id"]["videoId"]
    title = item["snippet"]["title"]
    channel = item["snippet"]["channelTitle"]

    # 📊 Video metadata
    video_response = youtube.videos().list(
        part="statistics,contentDetails,snippet",
        id=video_id
    ).execute()

    video_info = video_response["items"][0]
    stats = video_info["statistics"]
    details = video_info["contentDetails"]
    snippet = video_info["snippet"]

    videos_data.append({
    "video_id": video_id,
    "video_title": title,
    "description": snippet.get("description", ""),
    "channel_name": channel,
    "views": int(stats.get("viewCount", 0)),
    "likes": int(stats.get("likeCount", 0)),
    "duration": details["duration"],
    "publish_date": snippet["publishedAt"],
    "video_url": f"https://www.youtube.com/watch?v={video_id}"
})

    # 💬 Comments
    try:
        comment_response = youtube.commentThreads().list(
            part="snippet",
            videoId=video_id,
            maxResults=MAX_COMMENTS,
            textFormat="plainText"
        ).execute()

        for c in comment_response.get("items", []):
            comment = c["snippet"]["topLevelComment"]["snippet"]
            comments_data.append({
                "video_id": video_id,
                "comment_id": c["id"],
                "comment_text": comment["textDisplay"],
                "timestamp": comment["publishedAt"]
            })

    except:
        print(f"Comments disabled for video {video_id}")

# 💾 Save data
videos_df = pd.DataFrame(videos_data)
comments_df = pd.DataFrame(comments_data)

videos_df.to_csv("youtube_videos.csv", index=False)
comments_df.to_csv("youtube_comments.csv", index=False)

print("Phase 1 completed")
print("Videos collected:", len(videos_df))
print("Comments collected:", len(comments_df))


Phase 1 completed
Videos collected: 20
Comments collected: 1788


In [3]:
import pandas as pd
import re

# Load raw comments
comments_df = pd.read_csv("youtube_comments.csv")

# Text cleaning function
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)           # remove URLs
    text = re.sub(r"[^a-z0-9\s]", " ", text)      # remove special chars
    text = re.sub(r"\s+", " ", text).strip()      # normalize spaces
    return text

comments_df["clean_comment"] = comments_df["comment_text"].astype(str).apply(clean_text)
comments_df["comment_length"] = comments_df["clean_comment"].apply(lambda x: len(x.split()))

# Save cleaned comments
comments_df.to_csv("youtube_comments_cleaned.csv", index=False)

print("Phase 2 completed")
print("Total cleaned comments:", len(comments_df))
comments_df.head()


Phase 2 completed
Total cleaned comments: 1788


,video_id,comment_id,comment_text,timestamp,clean_comment,comment_length
0,l8W8fsDDKCw,UgySKbnakgiFZJb4wWh4AaABAg,Material ni మన Website నుంచి Download cheskond...,2026-09-09T07:25:13Z,material ni website download cheskondi website...,7
1,l8W8fsDDKCw,UgzMOvqitznO8AtVAuN4AaABAg,Thanks bro,2026-09-24T12:51:33Z,thanks bro,2
2,l8W8fsDDKCw,UgztH4Z54dpESNebJSN4AaABAg,Who else is watching this with a smile,2026-09-20T05:41:30Z,who else is watching this with a smile,8
3,l8W8fsDDKCw,UgzJpgicN6VhZ2KN5KZ4AaABAg,Sir python With Kuda Start cheyandi,2026-09-19T15:23:47Z,sir python with kuda start cheyandi,6
4,l8W8fsDDKCw,Ugym98VoXfSmqJdDkW54AaABAg,Sir\nAccessibility course telugu lo cheyandi Sir,2026-09-17T09:27:11Z,sir accessibility course telugu lo cheyandi sir,7


In [4]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer


In [5]:
comments_df = pd.read_csv("/content/youtube_comments_cleaned.csv")

# Sanity check
assert "clean_comment" in comments_df.columns
semantic_query = "Complete Python programming tutorial explaining concepts and problem solving"
model = SentenceTransformer("intfloat/e5-base-v2")
comments = comments_df["clean_comment"].fillna("").tolist()

comment_embeddings = model.encode(
    ["passage: " + c for c in comments],
    normalize_embeddings=True,
    batch_size=32,
    show_progress_bar=True
)
query_embedding = model.encode(
    ["query: " + semantic_query],
    normalize_embeddings=True
)
relevance_scores = np.dot(
    comment_embeddings,
    query_embedding.T
).squeeze()


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/67.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/56 [00:00<?, ?it/s]

In [6]:
comments_df["relevance_score"] = relevance_scores
comments_df = comments_df.sort_values(
    by="relevance_score",
    ascending=False
)
comments_df.to_csv(
    "phase3_comments_relevance.csv",
    index=False
)


In [7]:
phase3_df = pd.read_csv("phase3_comments_relevance.csv")
TOP_K = 1000  # justified as top-K semantic filtering

relevant_df = (
    phase3_df
    .sort_values("relevance_score", ascending=False)
    .head(TOP_K)
    .copy()
)

print("Selected relevant comments:", len(relevant_df))


Selected relevant comments: 1000


In [8]:
from transformers import pipeline

sentiment_model = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    truncation=True,
    max_length=512
)

print("Sentiment model loaded successfully")


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  501MB            

model.safetensors: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Sentiment model loaded successfully


In [9]:
sentiment_model("This course is absolutely amazing!")


[{'label': 'positive', 'score': 0.9776747822761536}]

In [10]:
def sentiment_score(text):
    result = sentiment_model(text)[0]

    label = result["label"].lower()
    score = result["score"]

    if label == "positive":
        return score
    elif label == "negative":
        return -score
    else:
        return 0.0


In [11]:
print(sentiment_score("This course is amazing"))
print(sentiment_score("This tutorial is terrible"))
print(sentiment_score("This is a Python video"))


0.9751406908035278
-0.9413253664970398
0.0


In [12]:
# Ensure clean_comment is string and not null
relevant_df = relevant_df.dropna(subset=["clean_comment"])
relevant_df["clean_comment"] = relevant_df["clean_comment"].astype(str)

print("Remaining relevant comments:", len(relevant_df))


Remaining relevant comments: 954


In [13]:
def safe_sentiment_score(text):
    try:
        return sentiment_score(text)
    except:
        return 0.0   # neutral fallback


In [14]:
relevant_df["sentiment_score"] = relevant_df["clean_comment"].apply(safe_sentiment_score)

relevant_df[["clean_comment", "relevance_score", "sentiment_score"]].head()


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


,clean_comment,relevance_score,sentiment_score
0,this is one of the best python tutorials i hav...,0.870045,0.981422
1,bro thank you really for this masterpiece this...,0.860903,0.984348
2,hi nana wow what a wonderful tutorial i refres...,0.859634,0.986517
3,this is most probably the best python tutorial...,0.857341,0.986081
4,guys here is a summurization used ai bytheway ...,0.854673,0.000000


In [15]:
relevant_df["sentiment_score"].describe()


,sentiment_score
count,954.000000
mean,0.243063
std,0.533067
min,-0.947956
25%,0.000000
50%,0.000000
75%,0.882375
max,0.990649


In [16]:
relevant_df = relevant_df.copy()
relevant_df.loc[:, "sentiment_score"] = relevant_df["clean_comment"].apply(safe_sentiment_score)


In [17]:
# Finalize Phase 4 output
relevant_df.to_csv(
    "phase4_relevant_comments_sentiment.csv",
    index=False
)

print("✅ Phase 4 finalized and saved successfully")
print("Rows saved:", len(relevant_df))


✅ Phase 4 finalized and saved successfully
Rows saved: 954


PHASE 5 – Emotion-Weighted Engagement Formulation (CORE NOVELTY)

In [18]:
import numpy as np
import pandas as pd
relevant_df = pd.read_csv("/content/phase4_relevant_comments_sentiment.csv")

relevant_df.head()
relevant_df["emotion_weighted_score"] = (
    relevant_df["relevance_score"] * relevant_df["sentiment_score"]
)


In [19]:
relevant_df["emotion_weighted_score"] = (
    relevant_df["relevance_score"] * relevant_df["sentiment_score"]
)


In [20]:
video_emotion_df = (
    relevant_df
    .groupby("video_id")
    .agg(
        emotional_engagement=("emotion_weighted_score", "mean"),
        emotional_variance=("emotion_weighted_score", "var"),
        relevant_comment_count=("emotion_weighted_score", "count")
    )
    .reset_index()
)


In [21]:
video_emotion_df.head()


,video_id,emotional_engagement,emotional_variance,relevant_comment_count
0,5B41CPvYMjY,-0.018382,0.095681,23
1,AQJbHAeUhzM,0.069792,0.101986,58
2,H2EJuAcrZYU,0.205295,0.251715,62
3,K5KVEU3aaeQ,0.222022,0.186886,55
4,Mf9GCn_LsUI,-0.057689,0.051890,54


In [22]:
video_emotion_df.to_csv("phase5_video_emotional_engagement.csv", index=False)
print("✅ Phase 5 completed successfully")


✅ Phase 5 completed successfully


PHASE 6: FINAL ENGAGEMENT SCORE & VIDEO RANKING

In [23]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Load Phase 5 data
phase5 = pd.read_csv("/content/phase5_video_emotional_engagement.csv")

# Load video metadata
videos = pd.read_csv("youtube_videos.csv")

# Merge on video_id
df = phase5.merge(videos, on="video_id", how="inner")

# Select columns for normalization
scaler = MinMaxScaler()

df[["likes_norm", "comment_count_norm"]] = scaler.fit_transform(
    df[["likes", "relevant_comment_count"]]
)

# Compute Final Engagement Score
df["final_engagement_score"] = (
    0.4 * df["emotional_engagement"] +
    0.2 * (1 - df["emotional_variance"]) +
    0.2 * df["likes_norm"] +
    0.2 * df["comment_count_norm"]
)

# Rank videos
df["rank"] = df["final_engagement_score"].rank(
    ascending=False, method="dense"
)

# Sort by rank
df = df.sort_values("rank")

# Save Phase 6 output
df[["video_id", "final_engagement_score", "rank"]].to_csv(
    "phase6_final_video_ranking.csv", index=False
)

df.head()
df.to_csv("phase6_full.csv", index=False)


PHASE 7 – Public Transcript Only Pipeline

single new phase 7

In [24]:
!pip install -q -U google-genai

from google import genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini client initialized successfully.")

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="Explain Python variables in one simple sentence."
)

print(response.text)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 23.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.1 which is incompatible.


Gemini client initialized successfully.
A Python variable is a named container used to store data that your program can easily remember, use, and update.


In [25]:
# ==========================================================
# PHASE 7 (FINAL - 20-VIDEO COHORT ALIGNED WITH PHASE 6)
# Intrinsic Teaching Quality Score (ITQS)
# ==========================================================

!pip install -q isodate google-genai

import pandas as pd
import numpy as np
import re
import isodate
import json
import time

from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import MinMaxScaler


# ==========================================================
# 1️⃣ Load Phase 6 Full Dataset
# ==========================================================

phase6_full = pd.read_csv(
    "phase6_full.csv"
)

print("Phase 6 Full rows:", len(phase6_full))
print(
    "Phase 6 Full unique videos:",
    phase6_full["video_id"].nunique()
)

if len(phase6_full) != 20:
    raise RuntimeError(
        f"Expected 20 videos in phase6_full.csv, "
        f"but found {len(phase6_full)}."
    )

if phase6_full["video_id"].nunique() != 20:
    raise RuntimeError(
        "Duplicate video IDs detected in phase6_full.csv."
    )


# ==========================================================
# 2️⃣ Create Phase 7 Dataset
# ==========================================================

videos_df = phase6_full[
    [
        "video_id",
        "video_title",
        "description",
        "duration"
    ]
].copy()

videos_df = videos_df.reset_index(drop=True)

print("\nTotal videos for Phase 7:", len(videos_df))

print("\nPhase 7 cohort:")
print(
    videos_df[
        ["video_id", "video_title"]
    ].to_string(index=False)
)


# ==========================================================
# 3️⃣ Combine Title + Description
# ==========================================================

videos_df["content_text"] = (
    videos_df["video_title"].fillna("") + " " +
    videos_df["description"].fillna("")
)


# ==========================================================
# 4️⃣ Semantic Alignment (E5)
# ==========================================================

print("\n========================================")
print("Loading E5 model")
print("========================================")

model_embed = SentenceTransformer(
    "intfloat/e5-base-v2"
)

semantic_query = (
    "Complete Python programming tutorial "
    "explaining concepts and problem solving"
)

query_embedding = model_embed.encode(
    ["query: " + semantic_query],
    normalize_embeddings=True
)

content_embeddings = model_embed.encode(
    [
        "passage: " + t
        for t in videos_df["content_text"]
    ],
    normalize_embeddings=True
)

videos_df["semantic_alignment"] = np.dot(
    content_embeddings,
    query_embedding.T
).squeeze()

print("✅ Semantic alignment calculated.")


# ==========================================================
# 5️⃣ Depth Score (Duration-based)
# ==========================================================

videos_df["duration_minutes"] = (
    videos_df["duration"].apply(
        lambda x:
        isodate.parse_duration(x)
        .total_seconds() / 60
    )
)

scaler = MinMaxScaler()

videos_df["depth_score"] = (
    scaler.fit_transform(
        videos_df[["duration_minutes"]]
    )
)

print("✅ Depth scores calculated.")


# ==========================================================
# 6️⃣ Lexical Diversity
# ==========================================================

def lexical_diversity(text):

    words = re.findall(
        r'\w+',
        text.lower()
    )

    if len(words) == 0:
        return 0

    return len(set(words)) / len(words)


videos_df["lexical_diversity"] = (
    videos_df["content_text"]
    .apply(lexical_diversity)
)

videos_df["lexical_diversity"] = (
    scaler.fit_transform(
        videos_df[["lexical_diversity"]]
    )
)

print("✅ Lexical diversity calculated.")


# ==========================================================
# 7️⃣ Gemini API
# ==========================================================

from google import genai
from google.colab import userdata


# ----------------------------------------------------------
# Load API key securely
# ----------------------------------------------------------

GEMINI_API_KEY = userdata.get(
    "GEMINI_API_KEY"
)

if not GEMINI_API_KEY:
    raise RuntimeError(
        "GEMINI_API_KEY not found in Colab Secrets."
    )


client = genai.Client(
    api_key=GEMINI_API_KEY
)


# ==========================================================
# 8️⃣ Batch LLM Teaching Score
# ==========================================================

BATCH_SIZE = 5
MAX_RETRIES = 5

all_scores = {}


print("\n========================================")
print("Starting LLM Teaching Score Evaluation")
print("========================================")

print(
    "Model: gemini-3.5-flash-lite"
)

print(
    "Total videos:",
    len(videos_df)
)


# ==========================================================
# Process videos in batches
# ==========================================================

for batch_start in range(
    0,
    len(videos_df),
    BATCH_SIZE
):

    batch_end = min(
        batch_start + BATCH_SIZE,
        len(videos_df)
    )

    batch_df = videos_df.iloc[
        batch_start:batch_end
    ]

    print(
        f"\nEvaluating videos "
        f"{batch_start} to {batch_end - 1}"
    )


    # ------------------------------------------------------
    # Build video information
    # ------------------------------------------------------

    video_blocks = ""

    for i, row in batch_df.iterrows():

        video_blocks += f"""
Video {i}
Title: {row['video_title']}
Description: {row['description']}
"""


    # ------------------------------------------------------
    # Prompt
    # ------------------------------------------------------

    prompt = f"""
Return ONLY valid JSON.
Do NOT include explanations.
Do NOT include markdown.
Do NOT include text before or after JSON.

Format:
{{ "0": number, "1": number, ... }}

Each number must be between 0 and 10.

Evaluate educational teaching quality using
the following six criteria:

1. Clear learning objective
2. Structured explanation
3. Logical concept progression
4. Example usage
5. Beginner friendliness
6. Practical problem solving

Give ONE overall teaching-quality score
from 0 to 10 for each video.

Videos:
{video_blocks}
"""


    # ------------------------------------------------------
    # Gemini Interactions API
    # ------------------------------------------------------

    raw_text = None

    for attempt in range(
        MAX_RETRIES
    ):

        try:

            print(
                f"Gemini request attempt "
                f"{attempt + 1}/{MAX_RETRIES}..."
            )

            interaction = (
                client.interactions.create(
                    model="gemini-3.5-flash-lite",
                    input=prompt
                )
            )

            raw_text = (
                interaction.output_text
                .strip()
            )

            print(
                "✅ Gemini response received."
            )

            break


        except Exception as e:

            print(
                f"⚠ Gemini error: {e}"
            )

            if attempt < MAX_RETRIES - 1:

                wait_time = min(
                    30,
                    2 ** attempt
                )

                print(
                    f"Waiting {wait_time} seconds "
                    f"before retry..."
                )

                time.sleep(
                    wait_time
                )

            else:

                raise RuntimeError(
                    f"Gemini failed for batch "
                    f"{batch_start}-{batch_end - 1} "
                    f"after {MAX_RETRIES} attempts."
                )


    # ------------------------------------------------------
    # Display response
    # ------------------------------------------------------

    print("\nRaw Gemini response:")
    print(raw_text[:1000])


    # ------------------------------------------------------
    # Extract JSON
    # ------------------------------------------------------

    json_match = re.search(
        r'\{.*\}',
        raw_text,
        re.DOTALL
    )

    if not json_match:

        raise RuntimeError(
            f"Valid JSON was not returned "
            f"for batch "
            f"{batch_start}-{batch_end - 1}."
        )


    json_text = json_match.group(0)


    # ------------------------------------------------------
    # Parse JSON
    # ------------------------------------------------------

    try:

        batch_scores = json.loads(
            json_text
        )

        print(
            "✅ JSON parsed successfully."
        )

    except json.JSONDecodeError as e:

        raise RuntimeError(
            f"JSON parsing failed for "
            f"batch "
            f"{batch_start}-{batch_end - 1}: "
            f"{e}"
        )


    # ------------------------------------------------------
    # Validate scores
    # ------------------------------------------------------

    expected_indices = list(
        batch_df.index
    )

    for idx in expected_indices:

        score = batch_scores.get(
            str(idx)
        )

        if score is None:

            raise RuntimeError(
                f"Missing teaching score "
                f"for video index {idx}."
            )


        try:

            score = float(score)

        except Exception:

            raise RuntimeError(
                f"Invalid teaching score "
                f"for video index {idx}: "
                f"{score}"
            )


        if not 0 <= score <= 10:

            raise RuntimeError(
                f"Teaching score must be "
                f"between 0 and 10. "
                f"Video {idx} received "
                f"{score}."
            )


        all_scores[str(idx)] = score


# ==========================================================
# 9️⃣ Assign Teaching Scores
# ==========================================================

videos_df["teaching_score_raw"] = (
    videos_df.index.map(
        lambda x:
        all_scores[str(x)]
    )
)


# ----------------------------------------------------------
# Normalize to 0–1
# ----------------------------------------------------------

videos_df["teaching_score"] = (
    videos_df["teaching_score_raw"] / 10
)


print("\n========================================")
print("TEACHING SCORES")
print("========================================")

print(
    videos_df[
        [
            "video_id",
            "video_title",
            "teaching_score_raw"
        ]
    ].to_string(index=False)
)

print(
    "\n✅ All 20 videos received valid "
    "LLM teaching scores."
)


# ==========================================================
# 🔟 Final Intrinsic Teaching Quality Score
# ==========================================================

videos_df["ITQS"] = (
    0.30 * videos_df["semantic_alignment"] +
    0.20 * videos_df["depth_score"] +
    0.20 * videos_df["lexical_diversity"] +
    0.30 * videos_df["teaching_score"]
)


# ==========================================================
# 1️⃣1️⃣ Final Validation
# ==========================================================

print("\n========================================")
print("FINAL PHASE 7 VALIDATION")
print("========================================")

print(
    "Phase 7 videos:",
    len(videos_df)
)

print(
    "Unique video IDs:",
    videos_df["video_id"].nunique()
)

print(
    "Missing teaching scores:",
    videos_df["teaching_score_raw"].isna().sum()
)

print(
    "Missing ITQS:",
    videos_df["ITQS"].isna().sum()
)


if (
    len(videos_df) != 20
    or videos_df["video_id"].nunique() != 20
    or videos_df["ITQS"].isna().any()
):

    raise RuntimeError(
        "Phase 7 validation failed."
    )


print(
    "✅ Phase 7 validation passed."
)


# ==========================================================
# 1️⃣2️⃣ Save Output
# ==========================================================

videos_df.to_csv(
    "phase7_intrinsic_teaching_quality.csv",
    index=False
)

print(
    "\n✅ PHASE 7 FINAL VERSION COMPLETED"
)

print(
    videos_df[
        ["video_id", "ITQS"]
    ].to_string(index=False)
)

Phase 6 Full rows: 20
Phase 6 Full unique videos: 20

Total videos for Phase 7: 20

Phase 7 cohort:
   video_id                                                                                       video_title
rfscVS0vtbw                                               Learn Python - Full Course for Beginners [Tutorial]
_uQrJ0TkZlc                                                                  Python Full Course for Beginners
t8pPdKYpowI                             Python Tutorial for Beginners - Learn Python in 5 Hours [FULL COURSE]
Rq5gJVxz55Q                                   Python Full Course for Beginners (13 Hours) – From Zero to Hero
rM-jDeSgOQw                                      FREE Python Course for Beginners (13 HOURS) - Code With Josh
kqtD5dpn9C8                                         Python for Beginners - Learn Coding with Python in 1 Hour
nLRL_NcnK-4                   Harvard CS50’s Introduction to Programming with Python – Full University Course
wUSDVGivd-8         

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Semantic alignment calculated.
✅ Depth scores calculated.
✅ Lexical diversity calculated.

Starting LLM Teaching Score Evaluation
Model: gemini-3.5-flash-lite
Total videos: 20

Evaluating videos 0 to 4
Gemini request attempt 1/5...
✅ Gemini response received.

Raw Gemini response:
{
  "0": 9,
  "1": 9,
  "2": 9,
  "3": 9,
  "4": 9
}
✅ JSON parsed successfully.

Evaluating videos 5 to 9
Gemini request attempt 1/5...
✅ Gemini response received.

Raw Gemini response:
{
  "5": 9,
  "6": 9,
  "7": 8,
  "8": 9,
  "9": 8
}
✅ JSON parsed successfully.

Evaluating videos 10 to 14
Gemini request attempt 1/5...
✅ Gemini response received.

Raw Gemini response:
{
  "10": 9,
  "11": 2,
  "12": 7,
  "13": 9,
  "14": 3
}
✅ JSON parsed successfully.

Evaluating videos 15 to 19
Gemini request attempt 1/5...
✅ Gemini response received.

Raw Gemini response:
{
  "15": 2,
  "16": 9,
  "17": 10,
  "18": 8,
  "19": 6
}
✅ JSON parsed successfully.

TEACHING SCORES
   video_id                               

PHASE 8 — Adaptive Hybrid Ranking

new pahse 8


In [26]:
# ==========================================================
# PHASE 8 – FINAL BALANCED HYBRID RANKING
# 20-VIDEO COHORT-ALIGNED VERSION
# ==========================================================

import pandas as pd
import numpy as np


# ==========================================================
# 1️⃣ Load Required Phases
# ==========================================================

phase5 = pd.read_csv(
    "phase5_video_emotional_engagement.csv"
)

phase6 = pd.read_csv(
    "phase6_final_video_ranking.csv"
)

phase7 = pd.read_csv(
    "phase7_intrinsic_teaching_quality.csv"
)


# ==========================================================
# 2️⃣ Clean Video IDs
# ==========================================================

for df in [phase5, phase6, phase7]:

    df["video_id"] = (
        df["video_id"]
        .astype(str)
        .str.strip()
    )


# ==========================================================
# 3️⃣ Validate Input Cohorts
# ==========================================================

ids5 = set(phase5["video_id"])
ids6 = set(phase6["video_id"])
ids7 = set(phase7["video_id"])

print("=" * 60)
print("INPUT COHORT VALIDATION")
print("=" * 60)

print("Phase 5 videos:", len(ids5))
print("Phase 6 videos:", len(ids6))
print("Phase 7 videos:", len(ids7))

print(
    "\nPhase 5 ∩ Phase 6:",
    len(ids5 & ids6)
)

print(
    "Phase 6 ∩ Phase 7:",
    len(ids6 & ids7)
)

print(
    "Phase 5 ∩ Phase 7:",
    len(ids5 & ids7)
)


# ----------------------------------------------------------
# Strict 20-video validation
# ----------------------------------------------------------

if len(ids5) != 20:
    raise RuntimeError(
        f"Phase 5 should contain 20 unique videos, "
        f"but contains {len(ids5)}."
    )

if len(ids6) != 20:
    raise RuntimeError(
        f"Phase 6 should contain 20 unique videos, "
        f"but contains {len(ids6)}."
    )

if len(ids7) != 20:
    raise RuntimeError(
        f"Phase 7 should contain 20 unique videos, "
        f"but contains {len(ids7)}."
    )


if not (
    ids5 == ids6 == ids7
):

    print("\n❌ Cohort mismatch detected.")

    print(
        "\nPhase 5 but not Phase 6:",
        sorted(ids5 - ids6)
    )

    print(
        "Phase 6 but not Phase 5:",
        sorted(ids6 - ids5)
    )

    print(
        "\nPhase 6 but not Phase 7:",
        sorted(ids6 - ids7)
    )

    print(
        "Phase 7 but not Phase 6:",
        sorted(ids7 - ids6)
    )

    raise RuntimeError(
        "Phase 5, Phase 6 and Phase 7 "
        "do not contain the same 20-video cohort."
    )


print(
    "\n✅ Phase 5, Phase 6 and Phase 7 "
    "contain the exact same 20 videos."
)


# ==========================================================
# 4️⃣ Merge Data
# ==========================================================

df = phase6[
    [
        "video_id",
        "final_engagement_score",
        "rank"
    ]
].merge(
    phase5[
        [
            "video_id",
            "relevant_comment_count"
        ]
    ],
    on="video_id",
    how="inner"
).merge(
    phase7[
        [
            "video_id",
            "ITQS"
        ]
    ],
    on="video_id",
    how="inner"
)


print(
    "\nMerged videos:",
    len(df)
)


if len(df) != 20:
    raise RuntimeError(
        f"Expected 20 merged videos, "
        f"but obtained {len(df)}."
    )


# ==========================================================
# 5️⃣ Adaptive Alpha
# ==========================================================

df["engagement_strength"] = np.log1p(
    df["relevant_comment_count"]
)

k = 5

df["alpha"] = (
    df["engagement_strength"] /
    (
        df["engagement_strength"] + k
    )
)


# ==========================================================
# 6️⃣ Hybrid Score
# ==========================================================

df["hybrid_score"] = (
    df["alpha"] *
    df["final_engagement_score"]
    +
    (1 - df["alpha"]) *
    df["ITQS"]
)


# ==========================================================
# 7️⃣ Final Ranking
# ==========================================================

df["final_rank"] = (
    df["hybrid_score"]
    .rank(
        ascending=False,
        method="dense"
    )
)

df = (
    df
    .sort_values(
        "final_rank"
    )
    .reset_index(drop=True)
)


# ==========================================================
# 8️⃣ Final Validation
# ==========================================================

print("\n========================================")
print("PHASE 8 FINAL VALIDATION")
print("========================================")

print(
    "Merged videos:",
    len(df)
)

print(
    "Unique video IDs:",
    df["video_id"].nunique()
)

print(
    "Missing hybrid scores:",
    df["hybrid_score"].isna().sum()
)

print(
    "Missing alpha values:",
    df["alpha"].isna().sum()
)

print(
    "ES-dominant:",
    (
        df["alpha"] * df["final_engagement_score"]
        >
        (1 - df["alpha"]) * df["ITQS"]
    ).sum()
)

print(
    "ITQS-dominant:",
    (
        df["alpha"] * df["final_engagement_score"]
        <=
        (1 - df["alpha"]) * df["ITQS"]
    ).sum()
)


if (
    len(df) != 20
    or df["video_id"].nunique() != 20
    or df["hybrid_score"].isna().any()
):

    raise RuntimeError(
        "Phase 8 validation failed."
    )


# ==========================================================
# 9️⃣ Save Final Ranking
# ==========================================================

df.to_csv(
    "phase8_hybrid_final_ranking.csv",
    index=False
)


# ==========================================================
# 🔟 Display Final Ranking
# ==========================================================

print(
    "\n✅ PHASE 8 COMPLETED SUCCESSFULLY"
)

print("\nFINAL TOP 10:")

print(
    df[
        [
            "video_id",
            "final_engagement_score",
            "ITQS",
            "alpha",
            "hybrid_score",
            "final_rank"
        ]
    ]
    .head(10)
    .to_string(index=False)
)

INPUT COHORT VALIDATION
Phase 5 videos: 20
Phase 6 videos: 20
Phase 7 videos: 20

Phase 5 ∩ Phase 6: 20
Phase 6 ∩ Phase 7: 20
Phase 5 ∩ Phase 7: 20

✅ Phase 5, Phase 6 and Phase 7 contain the exact same 20 videos.

Merged videos: 20

PHASE 8 FINAL VALIDATION
Merged videos: 20
Unique video IDs: 20
Missing hybrid scores: 0
Missing alpha values: 0
ES-dominant: 0
ITQS-dominant: 20

✅ PHASE 8 COMPLETED SUCCESSFULLY

FINAL TOP 10:
   video_id  final_engagement_score     ITQS    alpha  hybrid_score  final_rank
rfscVS0vtbw                0.643718 0.632674 0.441418      0.637549         1.0
_uQrJ0TkZlc                0.587764 0.629403 0.424552      0.611725         2.0
nLRL_NcnK-4                0.446214 0.740620 0.443764      0.609974         3.0
Rq5gJVxz55Q                0.489906 0.701892 0.444897      0.607580         4.0
rM-jDeSgOQw                0.487139 0.672896 0.454082      0.588547         5.0
t8pPdKYpowI                0.492384 0.628351 0.430796      0.569777         6.0
ix9cRaBkVe0

In [27]:
# ==========================================================
# PHASE 8 – BALANCED HYBRID RANKING (FINAL VERSION)
# ==========================================================

import pandas as pd
import numpy as np

# -------------------------------
# 1️⃣ Load Required Phases
# -------------------------------
phase5 = pd.read_csv("phase5_video_emotional_engagement.csv")
phase6 = pd.read_csv("phase6_final_video_ranking.csv")
phase7 = pd.read_csv("phase7_intrinsic_teaching_quality.csv")  # NEW ITQS

# -------------------------------
# 2️⃣ Merge Data
# -------------------------------
df = phase6.merge(
    phase5[["video_id", "relevant_comment_count"]],
    on="video_id",
    how="inner"
).merge(
    phase7[["video_id", "ITQS"]],
    on="video_id",
    how="inner"
)

print("Merged videos:", len(df))

# -------------------------------
# 3️⃣ Adaptive Alpha (Non-Linear)
# -------------------------------
# Engagement strength
df["engagement_strength"] = np.log1p(df["relevant_comment_count"])

# Balance tuning parameter
k = 5   # try values between 2–5 for tuning

df["alpha"] = df["engagement_strength"] / (
    df["engagement_strength"] + k
)

# -------------------------------
# 4️⃣ Hybrid Score
# -------------------------------
df["hybrid_score"] = (
    df["alpha"] * df["final_engagement_score"] +
    (1 - df["alpha"]) * df["ITQS"]
)

# -------------------------------
# 5️⃣ Final Ranking
# -------------------------------
df["final_rank"] = df["hybrid_score"].rank(
    ascending=False,
    method="dense"
)

df = df.sort_values("final_rank")

# -------------------------------
# 6️⃣ Save Output
# -------------------------------
df.to_csv("phase8_hybrid_final_ranking.csv", index=False)

print("✅ Phase 8 Completed Successfully")
print(df[["video_id", "hybrid_score", "final_rank"]].head())

Merged videos: 20
✅ Phase 8 Completed Successfully
      video_id  hybrid_score  final_rank
0  rfscVS0vtbw      0.637549         1.0
1  _uQrJ0TkZlc      0.611725         2.0
6  nLRL_NcnK-4      0.609974         3.0
3  Rq5gJVxz55Q      0.607580         4.0
4  rM-jDeSgOQw      0.588547         5.0


In [28]:
# Contribution analysis
df["es_contribution"] = df["alpha"] * df["final_engagement_score"]
df["itqs_contribution"] = (1 - df["alpha"]) * df["ITQS"]

print("ES dominant:", (df["es_contribution"] > df["itqs_contribution"]).sum())
print("ITQS dominant:", (df["itqs_contribution"] > df["es_contribution"]).sum())

print("\nAlpha distribution:")
print(df["alpha"].describe())

ES dominant: 0
ITQS dominant: 20

Alpha distribution:
count    20.000000
mean      0.427269
std       0.043874
min       0.293730
25%       0.428114
50%       0.444897
75%       0.449938
max       0.461013
Name: alpha, dtype: float64


In [29]:
from scipy.stats import spearmanr

hybrid_rank = df["hybrid_score"].rank(ascending=False)
es_rank = df["final_engagement_score"].rank(ascending=False)
itqs_rank = df["ITQS"].rank(ascending=False)

print("Hybrid vs ES:", spearmanr(hybrid_rank, es_rank))
print("Hybrid vs ITQS:", spearmanr(hybrid_rank, itqs_rank))

Hybrid vs ES: SignificanceResult(statistic=np.float64(0.8436090225563909), pvalue=np.float64(2.969127186415905e-06))
Hybrid vs ITQS: SignificanceResult(statistic=np.float64(0.8421052631578947), pvalue=np.float64(3.2174929874706926e-06))


LLm

In [30]:
import google.generativeai as genai
import os
import pandas as pd

# Set Google API key
genai.configure(api_key="REDACTED_API_KEY")

# Load model
model = genai.GenerativeModel("models/gemini-3.5-flash-lite")

phase5 = pd.read_csv("/content/phase5_video_emotional_engagement.csv")
phase6 = pd.read_csv("/content/phase6_final_video_ranking.csv")
phase7 = pd.read_csv("/content/phase7_intrinsic_teaching_quality.csv")
phase8 = pd.read_csv("/content/phase8_hybrid_final_ranking.csv")

# Merge safely
df = phase8.merge(phase6, on="video_id", how="inner") \
           .merge(phase7, on="video_id", how="inner") \
           .merge(phase5, on="video_id", how="inner")

print(df.columns)
df.head()

/usr/local/lib/python3.13/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Index(['video_id', 'final_engagement_score_x', 'rank_x',
       'relevant_comment_count_x', 'ITQS_x', 'engagement_strength', 'alpha',
       'hybrid_score', 'final_rank', 'final_engagement_score_y', 'rank_y',
       'video_title', 'description', 'duration', 'content_text',
       'semantic_alignment', 'duration_minutes', 'depth_score',
       'lexical_diversity', 'teaching_score_raw', 'teaching_score', 'ITQS_y',
       'emotional_engagement', 'emotional_variance',
       'relevant_comment_count_y'],
      dtype='object')


,video_id,final_engagement_score_x,rank_x,relevant_comment_count_x,ITQS_x,engagement_strength,alpha,hybrid_score,final_rank,final_engagement_score_y,...,semantic_alignment,duration_minutes,depth_score,lexical_diversity,teaching_score_raw,teaching_score,ITQS_y,emotional_engagement,emotional_variance,relevant_comment_count_y
0,rfscVS0vtbw,0.643718,1.0,51,0.632674,3.951244,0.441418,0.637549,1.0,0.643718,...,0.848276,266.866667,0.278512,0.262446,9.0,0.9,0.632674,0.400375,0.152687,51
1,_uQrJ0TkZlc,0.587764,2.0,39,0.629403,3.688879,0.424552,0.611725,2.0,0.587764,...,0.834812,374.116667,0.390505,0.154292,9.0,0.9,0.629403,0.308029,0.177236,39
2,nLRL_NcnK-4,0.446214,7.0,53,0.740620,3.988984,0.443764,0.609974,3.0,0.446214,...,0.830969,957.800000,1.000000,0.106649,9.0,0.9,0.740620,0.210381,0.094923,53
3,Rq5gJVxz55Q,0.489906,4.0,54,0.701892,4.007333,0.444897,0.607580,4.0,0.489906,...,0.824356,782.883333,0.817348,0.105580,9.0,0.9,0.701892,0.424274,0.150497,54
4,rM-jDeSgOQw,0.487139,5.0,63,0.672896,4.158883,0.454082,0.588547,5.0,0.487139,...,0.841493,612.416667,0.639343,0.112897,9.0,0.9,0.672896,0.369983,0.185030,63


In [31]:
print(df.columns)

Index(['video_id', 'final_engagement_score_x', 'rank_x',
       'relevant_comment_count_x', 'ITQS_x', 'engagement_strength', 'alpha',
       'hybrid_score', 'final_rank', 'final_engagement_score_y', 'rank_y',
       'video_title', 'description', 'duration', 'content_text',
       'semantic_alignment', 'duration_minutes', 'depth_score',
       'lexical_diversity', 'teaching_score_raw', 'teaching_score', 'ITQS_y',
       'emotional_engagement', 'emotional_variance',
       'relevant_comment_count_y'],
      dtype='object')


In [32]:
df_clean = df.copy()

df_clean["final_engagement_score"] = df_clean["final_engagement_score_x"]
df_clean["ITQS"] = df_clean["ITQS_x"]
df_clean["relevant_comment_count"] = df_clean["relevant_comment_count_x"]

df_clean = df_clean[
    [
        "video_id",
        "hybrid_score",
        "final_rank",
        "alpha",  # ← ADD THIS
        "final_engagement_score",
        "ITQS",
        "relevant_comment_count"
    ]
]

df_clean.head()

,video_id,hybrid_score,final_rank,alpha,final_engagement_score,ITQS,relevant_comment_count
0,rfscVS0vtbw,0.637549,1.0,0.441418,0.643718,0.632674,51
1,_uQrJ0TkZlc,0.611725,2.0,0.424552,0.587764,0.629403,39
2,nLRL_NcnK-4,0.609974,3.0,0.443764,0.446214,0.740620,53
3,Rq5gJVxz55Q,0.607580,4.0,0.444897,0.489906,0.701892,54
4,rM-jDeSgOQw,0.588547,5.0,0.454082,0.487139,0.672896,63


In [33]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [34]:
import time

def generate_explanation(row, retries=3):

    # Compute actual contributions
    es_contribution = row["alpha"] * row["final_engagement_score"]
    itqs_contribution = (1 - row["alpha"]) * row["ITQS"]

    dominant = "Engagement Score (ES)" if es_contribution > itqs_contribution else "Intrinsic Teaching Quality Score (ITQS)"

    prompt = f"""
    A hybrid ranking system computed the following:

    Hybrid Score = alpha * ES + (1 - alpha) * ITQS

    alpha: {row['alpha']:.3f}
    ES: {row['final_engagement_score']:.3f}
    ITQS: {row['ITQS']:.3f}

    ES contribution: {es_contribution:.3f}
    ITQS contribution: {itqs_contribution:.3f}

    Relevant Comment Count: {row['relevant_comment_count']}

    The dominant factor is: {dominant}

    Explain analytically why this video received its rank based ONLY on these values.
    Do not invent additional reasons.
    Keep explanation technical and concise.
    """

    for attempt in range(retries):
        try:
            response = model.generate_content(prompt)
            return response.text
        except Exception as e:
            print(f"Retry {attempt+1} due to error: {e}")
            time.sleep(2)

    return "Explanation generation failed."

In [35]:
top5 = df_clean.sort_values("final_rank").head(5).copy()

top5["explanation"] = top5.apply(generate_explanation, axis=1)

top5[["video_id", "final_rank", "explanation"]]

,video_id,final_rank,explanation
0,rfscVS0vtbw,1.0,"Based strictly on the provided values, the hyb..."
1,_uQrJ0TkZlc,2.0,"Based on the provided values, the hybrid ranki..."
2,nLRL_NcnK-4,3.0,"Based strictly on the provided values, the vid..."
3,Rq5gJVxz55Q,4.0,"Based on the provided values, the hybrid ranki..."
4,rM-jDeSgOQw,5.0,"Based on the provided values, the hybrid ranki..."


journal

In [38]:
# ============================================================
# PHASE 9A — MULTI-DOMAIN RETRIEVAL VALIDATION
# ============================================================
# Purpose:
#   Validate the framework on multiple educational domains.
#
# IMPORTANT:
#   - Do NOT modify Phase 1–8 cells.
#   - Do NOT overwrite original youtube_videos.csv,
#     youtube_comments.csv, or Phase 5–8 outputs.
#   - All revision data is saved under /content/revision_phase9a/
# ============================================================

!pip install -q google-api-python-client

from googleapiclient.discovery import build
from google.colab import userdata

import pandas as pd
import os
import re
import time
import json

# ------------------------------------------------------------
# 1. LOAD YOUTUBE API KEY FROM COLAB SECRET
# ------------------------------------------------------------

YOUTUBE_API_KEY = userdata.get("YOUTUBE_API_KEY")

if not YOUTUBE_API_KEY:
    raise RuntimeError(
        "YOUTUBE_API_KEY was not found in Colab Secrets. "
        "Add it under the 🔑 Secrets panel and run this cell again."
    )

youtube = build(
    "youtube",
    "v3",
    developerKey=YOUTUBE_API_KEY
)

print("YouTube API client initialized successfully.")


# ------------------------------------------------------------
# 2. EXPERIMENT CONFIGURATION
# ------------------------------------------------------------

QUERY_CONFIG = [
    {
        "query": "Full Python Course",
        "domain": "Programming",
        "slug": "python"
    },
    {
        "query": "Linear Algebra Course",
        "domain": "Mathematics",
        "slug": "linear_algebra"
    },
    {
        "query": "Quantum Mechanics Course",
        "domain": "Physics",
        "slug": "quantum_mechanics"
    },
    {
        "query": "Human Anatomy Course",
        "domain": "Biology",
        "slug": "human_anatomy"
    },
    {
        "query": "Machine Learning Course",
        "domain": "Data Science",
        "slug": "machine_learning"
    }
]

MAX_VIDEOS = 20
MAX_COMMENTS = 100

OUTPUT_ROOT = "/content/revision_phase9a"
os.makedirs(OUTPUT_ROOT, exist_ok=True)


# ------------------------------------------------------------
# 3. HELPER FUNCTIONS
# ------------------------------------------------------------

def parse_duration(duration):
    """
    Convert ISO 8601 YouTube duration into minutes.
    Example:
        PT1H20M30S -> 80.5 minutes
    """
    match = re.match(
        r"PT"
        r"(?:(\d+)H)?"
        r"(?:(\d+)M)?"
        r"(?:(\d+)S)?",
        duration
    )

    if not match:
        return 0.0

    hours = int(match.group(1) or 0)
    minutes = int(match.group(2) or 0)
    seconds = int(match.group(3) or 0)

    return hours * 60 + minutes + seconds / 60


def get_video_metadata(video_id):
    """
    Retrieve detailed metadata for one YouTube video.
    """

    try:
        response = youtube.videos().list(
            part="snippet,statistics,contentDetails",
            id=video_id
        ).execute()

        if not response.get("items"):
            return None

        item = response["items"][0]

        snippet = item.get("snippet", {})
        statistics = item.get("statistics", {})
        content = item.get("contentDetails", {})

        return {
            "video_id": video_id,
            "video_title": snippet.get("title", ""),
            "description": snippet.get("description", ""),
            "channel_name": snippet.get("channelTitle", ""),
            "views": int(statistics.get("viewCount", 0)),
            "likes": int(statistics.get("likeCount", 0)),
            "comment_count": int(statistics.get("commentCount", 0)),
            "duration": content.get("duration", ""),
            "duration_minutes": parse_duration(
                content.get("duration", "")
            ),
            "publish_date": snippet.get("publishedAt", ""),
            "video_url": f"https://www.youtube.com/watch?v={video_id}"
        }

    except Exception as e:
        print(f"Metadata error for {video_id}: {e}")
        return None


def get_comments(video_id, max_comments=100):
    """
    Retrieve top-level comments for one video.

    We intentionally keep this consistent with the original
    experimental design: up to 100 top-level comments/video.
    """

    comments = []

    try:
        response = youtube.commentThreads().list(
            part="snippet",
            videoId=video_id,
            maxResults=max_comments,
            textFormat="plainText"
        ).execute()

        for item in response.get("items", []):

            snippet = (
                item
                .get("snippet", {})
                .get("topLevelComment", {})
                .get("snippet", {})
            )

            comments.append({
                "video_id": video_id,
                "comment_id": (
                    item
                    .get("snippet", {})
                    .get("topLevelComment", {})
                    .get("id", "")
                ),
                "comment_text": snippet.get("textDisplay", ""),
                "comment_like_count": snippet.get("likeCount", 0),
                "comment_published_at": snippet.get(
                    "publishedAt", ""
                )
            })

    except Exception as e:
        # Comments can be disabled for some videos.
        print(
            f"  Comment retrieval skipped for {video_id}: "
            f"{type(e).__name__}"
        )

    return comments


# ------------------------------------------------------------
# 4. RUN RETRIEVAL FOR EACH EDUCATIONAL DOMAIN
# ------------------------------------------------------------

summary_rows = []

for config in QUERY_CONFIG:

    query = config["query"]
    domain = config["domain"]
    slug = config["slug"]

    print("\n" + "=" * 70)
    print(f"DOMAIN : {domain}")
    print(f"QUERY  : {query}")
    print("=" * 70)

    domain_dir = os.path.join(OUTPUT_ROOT, slug)
    os.makedirs(domain_dir, exist_ok=True)

    # --------------------------------------------------------
    # SEARCH FOR VIDEOS
    # --------------------------------------------------------

    search_response = youtube.search().list(
        q=query,
        part="id,snippet",
        type="video",
        maxResults=MAX_VIDEOS
    ).execute()

    search_items = search_response.get("items", [])

    video_ids = []

    for item in search_items:

        video_id = (
            item
            .get("id", {})
            .get("videoId")
        )

        if video_id:
            video_ids.append(video_id)

    # Remove duplicates while preserving order
    video_ids = list(dict.fromkeys(video_ids))

    print(f"Videos retrieved: {len(video_ids)}")

    # --------------------------------------------------------
    # VIDEO METADATA
    # --------------------------------------------------------

    video_rows = []

    for i, video_id in enumerate(video_ids, start=1):

        metadata = get_video_metadata(video_id)

        if metadata is not None:
            metadata["query"] = query
            metadata["domain"] = domain
            video_rows.append(metadata)

        print(
            f"  Metadata: {i}/{len(video_ids)}",
            end="\r"
        )

        time.sleep(0.05)

    print()

    videos_df = pd.DataFrame(video_rows)

    # --------------------------------------------------------
    # COMMENTS
    # --------------------------------------------------------

    comment_rows = []

    for i, video_id in enumerate(video_ids, start=1):

        comments = get_comments(
            video_id,
            max_comments=MAX_COMMENTS
        )

        for comment in comments:
            comment["query"] = query
            comment["domain"] = domain

        comment_rows.extend(comments)

        print(
            f"  Comments: {i}/{len(video_ids)}",
            end="\r"
        )

        time.sleep(0.05)

    print()

    comments_df = pd.DataFrame(comment_rows)

    # --------------------------------------------------------
    # SAVE DOMAIN-SPECIFIC FILES
    # --------------------------------------------------------

    videos_path = os.path.join(
        domain_dir,
        "videos.csv"
    )

    comments_path = os.path.join(
        domain_dir,
        "comments.csv"
    )

    videos_df.to_csv(
        videos_path,
        index=False
    )

    comments_df.to_csv(
        comments_path,
        index=False
    )

    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    unique_video_count = (
        videos_df["video_id"].nunique()
        if not videos_df.empty
        else 0
    )

    total_comments = len(comments_df)

    avg_comments = (
        total_comments / unique_video_count
        if unique_video_count > 0
        else 0
    )

    duplicate_video_ids = (
        len(videos_df) - unique_video_count
        if not videos_df.empty
        else 0
    )

    summary_rows.append({
        "query": query,
        "domain": domain,
        "slug": slug,
        "videos_retrieved": len(videos_df),
        "unique_videos": unique_video_count,
        "duplicate_video_ids": duplicate_video_ids,
        "total_comments": total_comments,
        "average_comments_per_video": avg_comments,
        "videos_file": videos_path,
        "comments_file": comments_path
    })

    print(f"\n{domain} validation:")
    print(f"  Videos              : {len(videos_df)}")
    print(f"  Unique videos       : {unique_video_count}")
    print(f"  Total comments      : {total_comments}")
    print(f"  Avg comments/video  : {avg_comments:.2f}")
    print(f"  Duplicate video IDs : {duplicate_video_ids}")


# ------------------------------------------------------------
# 5. CREATE OVERALL SUMMARY
# ------------------------------------------------------------

summary_df = pd.DataFrame(summary_rows)

summary_path = os.path.join(
    OUTPUT_ROOT,
    "query_summary.csv"
)

summary_df.to_csv(
    summary_path,
    index=False
)

print("\n" + "=" * 70)
print("PHASE 9A COMPLETE")
print("=" * 70)

display(summary_df)


# ------------------------------------------------------------
# 6. STRICT VALIDATION
# ------------------------------------------------------------

print("\nRunning final validation...\n")

validation_passed = True

for _, row in summary_df.iterrows():

    query = row["query"]
    videos = int(row["videos_retrieved"])
    unique_videos = int(row["unique_videos"])
    duplicates = int(row["duplicate_video_ids"])

    print(
        f"{query}: "
        f"{videos} videos | "
        f"{unique_videos} unique | "
        f"{duplicates} duplicates"
    )

    if videos != MAX_VIDEOS:
        validation_passed = False
        print(
            f"  WARNING: Expected {MAX_VIDEOS} videos "
            f"but retrieved {videos}."
        )

    if unique_videos != videos:
        validation_passed = False
        print("  WARNING: Duplicate video IDs detected.")


print("\n" + "=" * 70)

if validation_passed:
    print("✅ PHASE 9A VALIDATION PASSED")
    print("All five domains have the expected 20 unique videos.")
else:
    print("⚠️ PHASE 9A VALIDATION REQUIRES REVIEW")
    print("Check the warnings above before continuing.")

print("=" * 70)

print(f"\nAll Phase 9A files saved under:")
print(OUTPUT_ROOT)

YouTube API client initialized successfully.

DOMAIN : Programming
QUERY  : Full Python Course
Videos retrieved: 20
  Metadata: 20/20
  Comments: 20/20

Programming validation:
  Videos              : 20
  Unique videos       : 20
  Total comments      : 1952
  Avg comments/video  : 97.60
  Duplicate video IDs : 0

DOMAIN : Mathematics
QUERY  : Linear Algebra Course
Videos retrieved: 20
  Metadata: 20/20
  Comments: 20/20

Mathematics validation:
  Videos              : 20
  Unique videos       : 20
  Total comments      : 1759
  Avg comments/video  : 87.95
  Duplicate video IDs : 0

DOMAIN : Physics
QUERY  : Quantum Mechanics Course
Videos retrieved: 20
  Metadata: 20/20


  Comment retrieval skipped for uK2eFv7ne_Q: HttpError
  Comments: 20/20

Physics validation:
  Videos              : 20
  Unique videos       : 20
  Total comments      : 1846
  Avg comments/video  : 92.30
  Duplicate video IDs : 0

DOMAIN : Biology
QUERY  : Human Anatomy Course
Videos retrieved: 20

  Comments: 20/20

Biology validation:
  Videos              : 20
  Unique videos       : 20
  Total comments      : 1929
  Avg comments/video  : 96.45
  Duplicate video IDs : 0

DOMAIN : Data Science
QUERY  : Machine Learning Course
Videos retrieved: 20
  Metadata: 20/20
  Comments: 20/20

Data Science validation:
  Videos              : 20
  Unique videos       : 20
  Total comments      : 1534
  Avg comments/video  : 76.70
  Duplicate video IDs : 0

PHASE 9A COMPLETE


,query,domain,slug,videos_retrieved,unique_videos,duplicate_video_ids,total_comments,average_comments_per_video,videos_file,comments_file
0,Full Python Course,Programming,python,20,20,0,1952,97.60,/content/revision_phase9a/python/videos.csv,/content/revision_phase9a/python/comments.csv
1,Linear Algebra Course,Mathematics,linear_algebra,20,20,0,1759,87.95,/content/revision_phase9a/linear_algebra/video...,/content/revision_phase9a/linear_algebra/comme...
2,Quantum Mechanics Course,Physics,quantum_mechanics,20,20,0,1846,92.30,/content/revision_phase9a/quantum_mechanics/vi...,/content/revision_phase9a/quantum_mechanics/co...
3,Human Anatomy Course,Biology,human_anatomy,20,20,0,1929,96.45,/content/revision_phase9a/human_anatomy/videos...,/content/revision_phase9a/human_anatomy/commen...
4,Machine Learning Course,Data Science,machine_learning,20,20,0,1534,76.70,/content/revision_phase9a/machine_learning/vid...,/content/revision_phase9a/machine_learning/com...



Running final validation...

Full Python Course: 20 videos | 20 unique | 0 duplicates
Linear Algebra Course: 20 videos | 20 unique | 0 duplicates
Quantum Mechanics Course: 20 videos | 20 unique | 0 duplicates
Human Anatomy Course: 20 videos | 20 unique | 0 duplicates
Machine Learning Course: 20 videos | 20 unique | 0 duplicates

✅ PHASE 9A VALIDATION PASSED
All five domains have the expected 20 unique videos.

All Phase 9A files saved under:
/content/revision_phase9a


In [40]:
# ============================================================
# PHASE 9B — MULTI-DOMAIN PREPROCESSING + SEMANTIC RELEVANCE
# ============================================================
#
# Purpose:
#   Apply the same preprocessing + E5 semantic relevance
#   methodology across all five educational domains.
#
# Original experiment:
#   - lowercase text
#   - remove URLs
#   - remove non-alphanumeric characters
#   - normalize whitespace
#   - E5-base-v2 embeddings
#   - normalized embeddings
#   - dot-product similarity
#   - select top relevant comments
#
# IMPORTANT:
#   This does NOT modify Phase 1–8 files.
# ============================================================


# ------------------------------------------------------------
# 1. INSTALL / IMPORT
# ------------------------------------------------------------

!pip install -q sentence-transformers

import os
import re
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer


# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

INPUT_ROOT = "/content/revision_phase9a"
OUTPUT_ROOT = "/content/revision_phase9b"

os.makedirs(OUTPUT_ROOT, exist_ok=True)


# ------------------------------------------------------------
# 3. DOMAIN CONFIGURATION
# ------------------------------------------------------------

DOMAIN_CONFIG = [
    {
        "query": "Full Python Course",
        "domain": "Programming",
        "slug": "python",
        "semantic_query": (
            "Complete Python programming course "
            "explaining programming concepts and practical usage"
        )
    },
    {
        "query": "Linear Algebra Course",
        "domain": "Mathematics",
        "slug": "linear_algebra",
        "semantic_query": (
            "Complete linear algebra course "
            "explaining mathematical concepts, methods, and problem solving"
        )
    },
    {
        "query": "Quantum Mechanics Course",
        "domain": "Physics",
        "slug": "quantum_mechanics",
        "semantic_query": (
            "Complete quantum mechanics course "
            "explaining physical concepts, principles, and problem solving"
        )
    },
    {
        "query": "Human Anatomy Course",
        "domain": "Biology",
        "slug": "human_anatomy",
        "semantic_query": (
            "Complete human anatomy course "
            "explaining anatomical structures, functions, and biological concepts"
        )
    },
    {
        "query": "Machine Learning Course",
        "domain": "Data Science",
        "slug": "machine_learning",
        "semantic_query": (
            "Complete machine learning course "
            "explaining machine learning concepts, methods, and practical applications"
        )
    }
]

TOP_K = 1000


# ------------------------------------------------------------
# 4. LOAD E5 MODEL
# ------------------------------------------------------------

print("Loading E5 model...")

model = SentenceTransformer(
    "intfloat/e5-base-v2"
)

print("E5 model loaded successfully.")


# ------------------------------------------------------------
# 5. TEXT PREPROCESSING
# ------------------------------------------------------------

def clean_comment(text):
    """
    Same basic preprocessing strategy used in the
    original experiment.
    """

    if pd.isna(text):
        return ""

    text = str(text)

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(
        r"http\S+|www\S+|https\S+",
        "",
        text
    )

    # Remove non-alphanumeric characters
    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    # Normalize whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


# ------------------------------------------------------------
# 6. PROCESS EACH DOMAIN
# ------------------------------------------------------------

summary_rows = []

for config in DOMAIN_CONFIG:

    query = config["query"]
    domain = config["domain"]
    slug = config["slug"]
    semantic_query = config["semantic_query"]

    print("\n" + "=" * 75)
    print(f"DOMAIN          : {domain}")
    print(f"QUERY           : {query}")
    print(f"SEMANTIC QUERY  : {semantic_query}")
    print("=" * 75)

    # --------------------------------------------------------
    # INPUT
    # --------------------------------------------------------

    comments_path = os.path.join(
        INPUT_ROOT,
        slug,
        "comments.csv"
    )

    if not os.path.exists(comments_path):
        raise FileNotFoundError(
            f"Comments file not found:\n{comments_path}"
        )

    comments_df = pd.read_csv(
        comments_path
    )

    print(
        f"Raw comments loaded: {len(comments_df)}"
    )

    # --------------------------------------------------------
    # PREPROCESSING
    # --------------------------------------------------------

    comments_df["cleaned_comment"] = (
        comments_df["comment_text"]
        .apply(clean_comment)
    )

    comments_df["comment_length"] = (
        comments_df["cleaned_comment"]
        .str.split()
        .str.len()
    )

    # Remove empty comments
    before_count = len(comments_df)

    comments_df = comments_df[
        comments_df["cleaned_comment"].str.len() > 0
    ].copy()

    after_count = len(comments_df)

    removed_count = before_count - after_count

    print(
        f"After preprocessing : {after_count}"
    )

    print(
        f"Empty comments removed: {removed_count}"
    )

    # --------------------------------------------------------
    # E5 INPUT FORMAT
    # --------------------------------------------------------

    # E5 requires the appropriate prefixes.
    passage_texts = [
    "passage: " + text
    for text in comments_df["cleaned_comment"].tolist()
    ]

    query_text = (
        "query: "
        + semantic_query
    )

    # --------------------------------------------------------
    # QUERY EMBEDDING
    # --------------------------------------------------------

    query_embedding = model.encode(
        query_text,
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    # --------------------------------------------------------
    # COMMENT EMBEDDINGS
    # --------------------------------------------------------

    print("Generating comment embeddings...")

    comment_embeddings = model.encode(
        passage_texts,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    print(
        f"Embedding matrix shape: "
        f"{comment_embeddings.shape}"
    )

    # --------------------------------------------------------
    # DOT PRODUCT SIMILARITY
    # --------------------------------------------------------
    #
    # Because embeddings are L2-normalized,
    # dot product is equivalent to cosine similarity.
    # --------------------------------------------------------

    relevance_scores = (
        comment_embeddings @ query_embedding
    )

    comments_df["relevance_score"] = (
        relevance_scores
    )

    # --------------------------------------------------------
    # SORT BY RELEVANCE
    # --------------------------------------------------------

    comments_df = comments_df.sort_values(
        by="relevance_score",
        ascending=False
    ).reset_index(drop=True)

    # --------------------------------------------------------
    # TOP-K RELEVANT COMMENTS
    # --------------------------------------------------------

    actual_top_k = min(
        TOP_K,
        len(comments_df)
    )

    relevant_df = (
        comments_df
        .head(actual_top_k)
        .copy()
    )

    relevant_df["relevance_rank"] = (
        np.arange(1, len(relevant_df) + 1)
    )

    # Add experiment metadata
    relevant_df["query"] = query
    relevant_df["domain"] = domain
    relevant_df["semantic_query"] = semantic_query

    # --------------------------------------------------------
    # SAVE FULL CLEANED DATA
    # --------------------------------------------------------

    domain_output = os.path.join(
        OUTPUT_ROOT,
        slug
    )

    os.makedirs(
        domain_output,
        exist_ok=True
    )

    cleaned_path = os.path.join(
        domain_output,
        "comments_cleaned.csv"
    )

    comments_df.to_csv(
        cleaned_path,
        index=False
    )

    # --------------------------------------------------------
    # SAVE TOP-K RELEVANT COMMENTS
    # --------------------------------------------------------

    relevant_path = os.path.join(
        domain_output,
        "comments_relevant.csv"
    )

    relevant_df.to_csv(
        relevant_path,
        index=False
    )

    # --------------------------------------------------------
    # STATISTICS
    # --------------------------------------------------------

    summary_rows.append({
        "query": query,
        "domain": domain,
        "slug": slug,
        "raw_comments": before_count,
        "clean_comments": after_count,
        "empty_removed": removed_count,
        "relevant_comments": len(relevant_df),
        "mean_relevance": relevant_df[
            "relevance_score"
        ].mean(),
        "std_relevance": relevant_df[
            "relevance_score"
        ].std(),
        "min_relevance": relevant_df[
            "relevance_score"
        ].min(),
        "median_relevance": relevant_df[
            "relevance_score"
        ].median(),
        "max_relevance": relevant_df[
            "relevance_score"
        ].max(),
        "cleaned_file": cleaned_path,
        "relevant_file": relevant_path
    })

    # --------------------------------------------------------
    # DOMAIN OUTPUT
    # --------------------------------------------------------

    print("\nDomain results:")
    print(
        f"  Raw comments       : {before_count}"
    )
    print(
        f"  Clean comments     : {after_count}"
    )
    print(
        f"  Empty removed      : {removed_count}"
    )
    print(
        f"  Top relevant      : {len(relevant_df)}"
    )
    print(
        f"  Mean relevance     : "
        f"{relevant_df['relevance_score'].mean():.6f}"
    )
    print(
        f"  Std relevance      : "
        f"{relevant_df['relevance_score'].std():.6f}"
    )
    print(
        f"  Minimum relevance  : "
        f"{relevant_df['relevance_score'].min():.6f}"
    )
    print(
        f"  Median relevance   : "
        f"{relevant_df['relevance_score'].median():.6f}"
    )
    print(
        f"  Maximum relevance  : "
        f"{relevant_df['relevance_score'].max():.6f}"
    )

    # --------------------------------------------------------
    # SAMPLE
    # --------------------------------------------------------

    print("\nTop 5 relevant comments:")

    display(
        relevant_df[
            [
                "video_id",
                "relevance_rank",
                "relevance_score",
                "cleaned_comment"
            ]
        ].head(5)
    )


# ------------------------------------------------------------
# 7. SAVE OVERALL SUMMARY
# ------------------------------------------------------------

summary_df = pd.DataFrame(
    summary_rows
)

summary_path = os.path.join(
    OUTPUT_ROOT,
    "semantic_relevance_summary.csv"
)

summary_df.to_csv(
    summary_path,
    index=False
)


# ------------------------------------------------------------
# 8. FINAL VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("PHASE 9B — FINAL VALIDATION")
print("=" * 75)

validation_passed = True

for _, row in summary_df.iterrows():

    domain = row["domain"]

    raw_comments = int(
        row["raw_comments"]
    )

    clean_comments = int(
        row["clean_comments"]
    )

    relevant_comments = int(
        row["relevant_comments"]
    )

    print(
        f"\n{domain}"
    )

    print(
        f"  Raw comments      : {raw_comments}"
    )

    print(
        f"  Clean comments    : {clean_comments}"
    )

    print(
        f"  Relevant comments : {relevant_comments}"
    )

    # Basic integrity checks

    if clean_comments > raw_comments:
        validation_passed = False
        print(
            "  ❌ ERROR: Clean comments > raw comments"
        )

    if relevant_comments > clean_comments:
        validation_passed = False
        print(
            "  ❌ ERROR: Relevant comments > clean comments"
        )

    if relevant_comments == 0:
        validation_passed = False
        print(
            "  ❌ ERROR: No relevant comments"
        )


print("\n" + "=" * 75)

if validation_passed:
    print("✅ PHASE 9B VALIDATION PASSED")
    print(
        "All five domains successfully completed "
        "preprocessing and E5 semantic relevance."
    )
else:
    print("⚠️ PHASE 9B VALIDATION FAILED")
    print(
        "Inspect the errors above before continuing."
    )

print("=" * 75)

print(
    f"\nResults saved under:\n{OUTPUT_ROOT}"
)

print(
    f"\nSummary file:\n{summary_path}"
)

Loading E5 model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

E5 model loaded successfully.

DOMAIN          : Programming
QUERY           : Full Python Course
SEMANTIC QUERY  : Complete Python programming course explaining programming concepts and practical usage
Raw comments loaded: 1952
After preprocessing : 1910
Empty comments removed: 42
Generating comment embeddings...


Batches:   0%|          | 0/60 [00:00<?, ?it/s]

Embedding matrix shape: (1910, 768)

Domain results:
  Raw comments       : 1952
  Clean comments     : 1910
  Empty removed      : 42
  Top relevant      : 1000
  Mean relevance     : 0.781058
  Std relevance      : 0.029005
  Minimum relevance  : 0.743371
  Median relevance   : 0.773426
  Maximum relevance  : 0.893196

Top 5 relevant comments:


...


DOMAIN          : Data Science
QUERY           : Machine Learning Course
SEMANTIC QUERY  : Complete machine learning course explaining machine learning concepts, methods, and practical applications
Raw comments loaded: 1534
After preprocessing : 1495
Empty comments removed: 39
Generating comment embeddings...


Batches:   0%|          | 0/47 [00:00<?, ?it/s]

Embedding matrix shape: (1495, 768)

Domain results:
  Raw comments       : 1534
  Clean comments     : 1495
  Empty removed      : 39
  Top relevant      : 1000
  Mean relevance     : 0.778992
  Std relevance      : 0.028914
  Minimum relevance  : 0.737912
  Median relevance   : 0.774812
  Maximum relevance  : 0.898897

Top 5 relevant comments:


,video_id,relevance_rank,relevance_score,cleaned_comment
0,7gEzEAIELgQ,1,0.898897,want to know full course about machine learning
1,hR-tMLTMw0s,2,0.880507,best course for machine learning
2,7gEzEAIELgQ,3,0.871889,machine learning full course video poduga plss...
3,adhgNQmPXZU,4,0.871424,ml course please
4,7IgVGSaQPaw,5,0.870246,full course for ml



PHASE 9B — FINAL VALIDATION

Programming
  Raw comments      : 1952
  Clean comments    : 1910
  Relevant comments : 1000

Mathematics
  Raw comments      : 1759
  Clean comments    : 1716
  Relevant comments : 1000

Physics
  Raw comments      : 1846
  Clean comments    : 1804
  Relevant comments : 1000

Biology
  Raw comments      : 1929
  Clean comments    : 1876
  Relevant comments : 1000

Data Science
  Raw comments      : 1534
  Clean comments    : 1495
  Relevant comments : 1000

✅ PHASE 9B VALIDATION PASSED
All five domains successfully completed preprocessing and E5 semantic relevance.

Results saved under:
/content/revision_phase9b

Summary file:
/content/revision_phase9b/semantic_relevance_summary.csv


In [41]:
# ============================================================
# PHASE 9C — MULTI-DOMAIN SENTIMENT + EMOTIONAL ENGAGEMENT
# ============================================================
#
# Input:
#   Phase 9B top-1000 relevant comments/domain
#
# Output:
#   Video-level emotional engagement statistics
#
# Same methodology as original experiment:
#
#   positive sentiment -> positive score
#   negative sentiment -> negative score
#   neutral sentiment  -> 0
#
#   emotion_weighted_score =
#       relevance_score * sentiment_score
#
#   emotional_engagement =
#       mean(emotion_weighted_score)
#
#   emotional_variance =
#       variance(emotion_weighted_score)
#
# ============================================================


# ------------------------------------------------------------
# 1. INSTALL / IMPORT
# ------------------------------------------------------------

!pip install -q transformers torch pandas numpy tqdm

import os
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)


# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

INPUT_ROOT = "/content/revision_phase9b"
OUTPUT_ROOT = "/content/revision_phase9c"

os.makedirs(
    OUTPUT_ROOT,
    exist_ok=True
)


# ------------------------------------------------------------
# 3. DOMAIN CONFIGURATION
# ------------------------------------------------------------

DOMAIN_CONFIG = [
    {
        "query": "Full Python Course",
        "domain": "Programming",
        "slug": "python"
    },
    {
        "query": "Linear Algebra Course",
        "domain": "Mathematics",
        "slug": "linear_algebra"
    },
    {
        "query": "Quantum Mechanics Course",
        "domain": "Physics",
        "slug": "quantum_mechanics"
    },
    {
        "query": "Human Anatomy Course",
        "domain": "Biology",
        "slug": "human_anatomy"
    },
    {
        "query": "Machine Learning Course",
        "domain": "Data Science",
        "slug": "machine_learning"
    }
]


# ------------------------------------------------------------
# 4. LOAD CARDIFFNLP SENTIMENT MODEL
# ------------------------------------------------------------

SENTIMENT_MODEL_NAME = (
    "cardiffnlp/twitter-roberta-base-sentiment-latest"
)

print("Loading CardiffNLP RoBERTa sentiment model...")

tokenizer = AutoTokenizer.from_pretrained(
    SENTIMENT_MODEL_NAME
)

sentiment_model = AutoModelForSequenceClassification.from_pretrained(
    SENTIMENT_MODEL_NAME
)

# Select GPU if available
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

sentiment_model = sentiment_model.to(device)
sentiment_model.eval()

print(
    f"Sentiment model loaded successfully."
)

print(
    f"Device: {device}"
)


# ------------------------------------------------------------
# 5. IDENTIFY SENTIMENT LABELS
# ------------------------------------------------------------

id2label = sentiment_model.config.id2label

print("\nModel labels:")

for idx, label in id2label.items():
    print(
        f"  {idx} -> {label}"
    )


# ------------------------------------------------------------
# 6. SENTIMENT SCORING FUNCTION
# ------------------------------------------------------------

def sentiment_score_from_probabilities(
    probabilities
):
    """
    Convert RoBERTa class probabilities into
    the scalar sentiment score used by the
    original pipeline.

    Positive:
        +P(positive)

    Negative:
        -P(negative)

    Neutral:
        0

    The model probabilities are first mapped
    according to the model's label names.
    """

    positive_prob = 0.0
    negative_prob = 0.0

    for idx, label in id2label.items():

        label_lower = str(label).lower()

        if "positive" in label_lower:
            positive_prob = float(
                probabilities[idx]
            )

        elif "negative" in label_lower:
            negative_prob = float(
                probabilities[idx]
            )

    return positive_prob - negative_prob


# ------------------------------------------------------------
# 7. BATCH SENTIMENT PREDICTION
# ------------------------------------------------------------

def predict_sentiment(
    texts,
    batch_size=32
):
    """
    Generate sentiment scores and labels
    for a list of comments.
    """

    all_scores = []
    all_labels = []
    all_positive = []
    all_negative = []
    all_neutral = []

    for start in tqdm(
        range(
            0,
            len(texts),
            batch_size
        ),
        desc="Sentiment batches"
    ):

        batch_texts = texts[
            start:start + batch_size
        ]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        )

        encoded = {
            key: value.to(device)
            for key, value in encoded.items()
        }

        with torch.no_grad():

            outputs = sentiment_model(
                **encoded
            )

            probabilities = torch.softmax(
                outputs.logits,
                dim=1
            ).cpu().numpy()

        for probs in probabilities:

            score = (
                sentiment_score_from_probabilities(
                    probs
                )
            )

            predicted_idx = int(
                np.argmax(probs)
            )

            predicted_label = str(
                id2label[predicted_idx]
            )

            # Store probability values
            positive_prob = 0.0
            negative_prob = 0.0
            neutral_prob = 0.0

            for idx, label in id2label.items():

                label_lower = str(label).lower()

                if "positive" in label_lower:
                    positive_prob = float(
                        probs[idx]
                    )

                elif "negative" in label_lower:
                    negative_prob = float(
                        probs[idx]
                    )

                elif "neutral" in label_lower:
                    neutral_prob = float(
                        probs[idx]
                    )

            all_scores.append(score)
            all_labels.append(predicted_label)
            all_positive.append(positive_prob)
            all_negative.append(negative_prob)
            all_neutral.append(neutral_prob)

    return (
        all_scores,
        all_labels,
        all_positive,
        all_negative,
        all_neutral
    )


# ------------------------------------------------------------
# 8. PROCESS ALL FIVE DOMAINS
# ------------------------------------------------------------

domain_summary_rows = []

all_video_emotion_rows = []

for config in DOMAIN_CONFIG:

    query = config["query"]
    domain = config["domain"]
    slug = config["slug"]

    print("\n" + "=" * 75)
    print(f"DOMAIN : {domain}")
    print(f"QUERY  : {query}")
    print("=" * 75)

    # --------------------------------------------------------
    # INPUT
    # --------------------------------------------------------

    input_path = os.path.join(
        INPUT_ROOT,
        slug,
        "comments_relevant.csv"
    )

    if not os.path.exists(input_path):

        raise FileNotFoundError(
            f"Phase 9B relevant-comments file not found:\n"
            f"{input_path}"
        )

    relevant_df = pd.read_csv(
        input_path
    )

    print(
        f"Relevant comments loaded: "
        f"{len(relevant_df)}"
    )

    # --------------------------------------------------------
    # CHECK REQUIRED COLUMNS
    # --------------------------------------------------------

    required_columns = [
        "video_id",
        "cleaned_comment",
        "relevance_score"
    ]

    missing_columns = [
        col
        for col in required_columns
        if col not in relevant_df.columns
    ]

    if missing_columns:

        raise ValueError(
            f"Missing required columns: "
            f"{missing_columns}"
        )

    # --------------------------------------------------------
    # SENTIMENT
    # --------------------------------------------------------

    texts = (
        relevant_df[
            "cleaned_comment"
        ]
        .fillna("")
        .astype(str)
        .tolist()
    )

    (
        sentiment_scores,
        sentiment_labels,
        positive_probs,
        negative_probs,
        neutral_probs
    ) = predict_sentiment(
        texts,
        batch_size=32
    )

    relevant_df[
        "sentiment_score"
    ] = sentiment_scores

    relevant_df[
        "sentiment_label"
    ] = sentiment_labels

    relevant_df[
        "positive_probability"
    ] = positive_probs

    relevant_df[
        "negative_probability"
    ] = negative_probs

    relevant_df[
        "neutral_probability"
    ] = neutral_probs

    # --------------------------------------------------------
    # RELEVANCE-WEIGHTED SENTIMENT
    # --------------------------------------------------------

    relevant_df[
        "emotion_weighted_score"
    ] = (
        relevant_df["relevance_score"]
        *
        relevant_df["sentiment_score"]
    )

    # Add metadata
    relevant_df["query"] = query
    relevant_df["domain"] = domain

    # --------------------------------------------------------
    # SAVE COMMENT-LEVEL SENTIMENT
    # --------------------------------------------------------

    domain_output = os.path.join(
        OUTPUT_ROOT,
        slug
    )

    os.makedirs(
        domain_output,
        exist_ok=True
    )

    comment_output_path = os.path.join(
        domain_output,
        "relevant_comments_sentiment.csv"
    )

    relevant_df.to_csv(
        comment_output_path,
        index=False
    )

    # --------------------------------------------------------
    # VIDEO-LEVEL AGGREGATION
    # --------------------------------------------------------

    video_emotion_df = (
        relevant_df
        .groupby("video_id")
        .agg(
            emotional_engagement=(
                "emotion_weighted_score",
                "mean"
            ),
            emotional_variance=(
                "emotion_weighted_score",
                "var"
            ),
            relevant_comment_count=(
                "emotion_weighted_score",
                "count"
            )
        )
        .reset_index()
    )

    # Variance is undefined for a single observation.
    # Replace such cases with zero.
    video_emotion_df[
        "emotional_variance"
    ] = (
        video_emotion_df[
            "emotional_variance"
        ]
        .fillna(0)
    )

    video_emotion_df["query"] = query
    video_emotion_df["domain"] = domain

    # --------------------------------------------------------
    # SENTIMENT DISTRIBUTION
    # --------------------------------------------------------

    sentiment_distribution = (
        relevant_df[
            "sentiment_label"
        ]
        .value_counts()
        .to_dict()
    )

    positive_count = sum(
        1
        for label in relevant_df[
            "sentiment_label"
        ]
        if "positive" in str(label).lower()
    )

    negative_count = sum(
        1
        for label in relevant_df[
            "sentiment_label"
        ]
        if "negative" in str(label).lower()
    )

    neutral_count = sum(
        1
        for label in relevant_df[
            "sentiment_label"
        ]
        if "neutral" in str(label).lower()
    )

    # --------------------------------------------------------
    # SAVE VIDEO-LEVEL RESULTS
    # --------------------------------------------------------

    video_output_path = os.path.join(
        domain_output,
        "video_emotional_engagement.csv"
    )

    video_emotion_df.to_csv(
        video_output_path,
        index=False
    )

    # Add to combined result
    all_video_emotion_rows.append(
        video_emotion_df
    )

    # --------------------------------------------------------
    # DOMAIN SUMMARY
    # --------------------------------------------------------

    domain_summary_rows.append({
        "query": query,
        "domain": domain,
        "slug": slug,
        "relevant_comments": len(relevant_df),
        "videos_with_relevant_comments": (
            video_emotion_df[
                "video_id"
            ].nunique()
        ),
        "positive_comments": positive_count,
        "negative_comments": negative_count,
        "neutral_comments": neutral_count,
        "positive_pct": (
            positive_count /
            len(relevant_df) * 100
        ),
        "negative_pct": (
            negative_count /
            len(relevant_df) * 100
        ),
        "neutral_pct": (
            neutral_count /
            len(relevant_df) * 100
        ),
        "mean_sentiment": (
            relevant_df[
                "sentiment_score"
            ].mean()
        ),
        "std_sentiment": (
            relevant_df[
                "sentiment_score"
            ].std()
        ),
        "mean_emotional_engagement": (
            video_emotion_df[
                "emotional_engagement"
            ].mean()
        ),
        "mean_emotional_variance": (
            video_emotion_df[
                "emotional_variance"
            ].mean()
        ),
        "comment_output": comment_output_path,
        "video_output": video_output_path
    })

    # --------------------------------------------------------
    # PRINT RESULTS
    # --------------------------------------------------------

    print("\nSentiment distribution:")

    print(
        f"  Positive : {positive_count}"
        f" ({positive_count / len(relevant_df) * 100:.2f}%)"
    )

    print(
        f"  Negative : {negative_count}"
        f" ({negative_count / len(relevant_df) * 100:.2f}%)"
    )

    print(
        f"  Neutral  : {neutral_count}"
        f" ({neutral_count / len(relevant_df) * 100:.2f}%)"
    )

    print(
        f"\nMean sentiment: "
        f"{relevant_df['sentiment_score'].mean():.6f}"
    )

    print(
        f"Std sentiment : "
        f"{relevant_df['sentiment_score'].std():.6f}"
    )

    print(
        f"\nVideos with relevant comments: "
        f"{video_emotion_df['video_id'].nunique()}"
    )

    print(
        f"Mean emotional engagement: "
        f"{video_emotion_df['emotional_engagement'].mean():.6f}"
    )

    print(
        f"Mean emotional variance: "
        f"{video_emotion_df['emotional_variance'].mean():.6f}"
    )

    print("\nTop 5 videos by emotional engagement:")

    display(
        video_emotion_df
        .sort_values(
            "emotional_engagement",
            ascending=False
        )
        .head(5)
    )


# ------------------------------------------------------------
# 9. COMBINE ALL DOMAINS
# ------------------------------------------------------------

combined_emotion_df = pd.concat(
    all_video_emotion_rows,
    ignore_index=True
)

combined_output_path = os.path.join(
    OUTPUT_ROOT,
    "all_domains_video_emotional_engagement.csv"
)

combined_emotion_df.to_csv(
    combined_output_path,
    index=False
)


# ------------------------------------------------------------
# 10. DOMAIN SUMMARY
# ------------------------------------------------------------

domain_summary_df = pd.DataFrame(
    domain_summary_rows
)

summary_output_path = os.path.join(
    OUTPUT_ROOT,
    "sentiment_emotional_engagement_summary.csv"
)

domain_summary_df.to_csv(
    summary_output_path,
    index=False
)


# ------------------------------------------------------------
# 11. FINAL VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("PHASE 9C — FINAL VALIDATION")
print("=" * 75)

validation_passed = True

for _, row in domain_summary_df.iterrows():

    domain = row["domain"]

    relevant_comments = int(
        row["relevant_comments"]
    )

    video_count = int(
        row["videos_with_relevant_comments"]
    )

    print(f"\n{domain}")
    print(
        f"  Relevant comments : "
        f"{relevant_comments}"
    )
    print(
        f"  Videos             : "
        f"{video_count}"
    )

    if relevant_comments != 1000:

        validation_passed = False

        print(
            "  ⚠️ WARNING: Expected "
            "1000 relevant comments."
        )

    if video_count == 0:

        validation_passed = False

        print(
            "  ❌ ERROR: No videos found."
        )

    if not np.isfinite(
        row["mean_sentiment"]
    ):

        validation_passed = False

        print(
            "  ❌ ERROR: Invalid sentiment value."
        )


print("\n" + "=" * 75)

print(
    f"Total video-level records: "
    f"{len(combined_emotion_df)}"
)

print(
    f"Unique video IDs: "
    f"{combined_emotion_df['video_id'].nunique()}"
)

print("=" * 75)

if validation_passed:

    print(
        "✅ PHASE 9C VALIDATION PASSED"
    )

    print(
        "All five domains completed "
        "sentiment and emotional-engagement analysis."
    )

else:

    print(
        "⚠️ PHASE 9C VALIDATION REQUIRES REVIEW"
    )

print("=" * 75)

print(
    f"\nResults saved under:\n"
    f"{OUTPUT_ROOT}"
)

print(
    f"\nCombined video-level file:\n"
    f"{combined_output_path}"
)

print(
    f"\nSummary file:\n"
    f"{summary_output_path}"
)

Loading CardiffNLP RoBERTa sentiment model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sentiment model loaded successfully.
Device: cuda

Model labels:
  0 -> negative
  1 -> neutral
  2 -> positive

DOMAIN : Programming
QUERY  : Full Python Course
Relevant comments loaded: 1000


Sentiment batches:   0%|          | 0/32 [00:00<?, ?it/s]

...


DOMAIN : Data Science
QUERY  : Machine Learning Course
Relevant comments loaded: 1000


Sentiment batches:   0%|          | 0/32 [00:00<?, ?it/s]


Sentiment distribution:
  Positive : 389 (38.90%)
  Negative : 59 (5.90%)
  Neutral  : 552 (55.20%)

Mean sentiment: 0.332410
Std sentiment : 0.484544

Videos with relevant comments: 20
Mean emotional engagement: 0.251309
Mean emotional variance: 0.133257

Top 5 videos by emotional engagement:


,video_id,emotional_engagement,emotional_variance,relevant_comment_count,query,domain
11,b2q5OFtxm6A,0.436605,0.138882,53,Machine Learning Course,Data Science
10,adhgNQmPXZU,0.431619,0.088315,74,Machine Learning Course,Data Science
18,sbmPMtQkjxY,0.415542,0.113881,3,Machine Learning Course,Data Science
4,GwIo3gDZCVQ,0.408797,0.141279,92,Machine Learning Course,Data Science
14,hR-tMLTMw0s,0.397639,0.149594,71,Machine Learning Course,Data Science



PHASE 9C — FINAL VALIDATION

Programming
  Relevant comments : 1000
  Videos             : 20

Mathematics
  Relevant comments : 1000
  Videos             : 20

Physics
  Relevant comments : 1000
  Videos             : 19

Biology
  Relevant comments : 1000
  Videos             : 20

Data Science
  Relevant comments : 1000
  Videos             : 20

Total video-level records: 99
Unique video IDs: 99
✅ PHASE 9C VALIDATION PASSED
All five domains completed sentiment and emotional-engagement analysis.

Results saved under:
/content/revision_phase9c

Combined video-level file:
/content/revision_phase9c/all_domains_video_emotional_engagement.csv

Summary file:
/content/revision_phase9c/sentiment_emotional_engagement_summary.csv


In [44]:
# ============================================================
# PHASE 9D — NUMERICAL PRECISION FIX
# ============================================================

fes_path = "/content/revision_phase9d/phase9d_fes_full.csv"

fes_df = pd.read_csv(fes_path)

# Numerical safety for Min-Max normalized values.
# This does NOT change the intended mathematical value.
fes_df["likes_norm"] = fes_df["likes_norm"].clip(0.0, 1.0)
fes_df["comment_count_norm"] = fes_df["comment_count_norm"].clip(0.0, 1.0)

# Recompute FES after the numerical safety correction.
fes_df["final_engagement_score"] = (
    0.4 * fes_df["emotional_engagement"]
    + 0.2 * (1.0 - fes_df["emotional_variance"])
    + 0.2 * fes_df["likes_norm"]
    + 0.2 * fes_df["comment_count_norm"]
)

# Recompute global rank
fes_df = fes_df.sort_values(
    "final_engagement_score",
    ascending=False
).reset_index(drop=True)

fes_df["global_fes_rank"] = range(
    1,
    len(fes_df) + 1
)

# Recompute domain rank
fes_df["domain_fes_rank"] = (
    fes_df
    .groupby("domain")["final_engagement_score"]
    .rank(
        method="first",
        ascending=False
    )
    .astype(int)
)

# Save corrected complete file
fes_df.to_csv(
    fes_path,
    index=False
)

# Save corrected Top 10
top10_path = "/content/revision_phase9d/phase9d_fes_top10.csv"

fes_df.head(10).to_csv(
    top10_path,
    index=False
)

# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

print("=" * 75)
print("PHASE 9D — CORRECTED VALIDATION")
print("=" * 75)

print(
    f"Video records       : {len(fes_df)}"
)

print(
    f"Unique video IDs    : {fes_df['video_id'].nunique()}"
)

print(
    f"Missing FES values  : "
    f"{fes_df['final_engagement_score'].isna().sum()}"
)

print(
    f"FES minimum         : "
    f"{fes_df['final_engagement_score'].min():.6f}"
)

print(
    f"FES maximum         : "
    f"{fes_df['final_engagement_score'].max():.6f}"
)

print(
    f"Likes norm range    : "
    f"{fes_df['likes_norm'].min():.6f} → "
    f"{fes_df['likes_norm'].max():.6f}"
)

print(
    f"Comments norm range : "
    f"{fes_df['comment_count_norm'].min():.6f} → "
    f"{fes_df['comment_count_norm'].max():.6f}"
)

print("\nTop 10 after correction:")

display(
    fes_df[
        [
            "global_fes_rank",
            "domain",
            "video_id",
            "video_title",
            "emotional_engagement",
            "emotional_variance",
            "likes_norm",
            "comment_count_norm",
            "final_engagement_score"
        ]
    ].head(10)
)

print("\n" + "=" * 75)
print("✅ PHASE 9D VALIDATION PASSED")
print("=" * 75)

print(
    "\nThe 1.0000000000000002 value was only "
    "floating-point precision and has been safely clipped to 1.0."
)

print(
    f"\nCorrected FES file:\n{fes_path}"
)

print(
    f"\nCorrected Top-10 file:\n{top10_path}"
)

PHASE 9D — CORRECTED VALIDATION
Video records       : 99
Unique video IDs    : 99
Missing FES values  : 0
FES minimum         : 0.096645
FES maximum         : 0.714660
Likes norm range    : 0.000000 → 1.000000
Comments norm range : 0.000000 → 1.000000

Top 10 after correction:


,global_fes_rank,domain,video_id,video_title,emotional_engagement,emotional_variance,likes_norm,comment_count_norm,final_engagement_score
0,1,Programming,_uQrJ0TkZlc,Python Full Course for Beginners,0.358922,0.144543,1.000000,1.000000,0.714660
1,2,Programming,rfscVS0vtbw,Learn Python - Full Course for Beginners [Tuto...,0.437677,0.140434,0.883062,0.733737,0.670344
2,3,Programming,XKHEtdqhLK8,Python Full Course for free 🐍,0.345352,0.141863,0.785238,0.447099,0.556236
3,4,Programming,UrsmFxEIp5k,Python Tutorial For Beginners in Hindi | Compl...,0.313187,0.178090,0.364740,0.966351,0.555875
4,5,Programming,kqtD5dpn9C8,Python for Beginners - Learn Coding with Pytho...,0.150772,0.076436,0.448496,0.458034,0.426328
5,6,Biology,8Z1C7B98KEY,11 Organ Systems of the Human Body (Made Easy!),0.561667,0.118335,0.013878,0.005380,0.404851
6,7,Programming,t8pPdKYpowI,Python Tutorial for Beginners - Learn Python i...,0.480334,0.168309,0.134692,0.065420,0.398494
7,8,Programming,ix9cRaBkVe0,Python Full Course for free 🐍,0.217516,0.182591,0.287094,0.357533,0.379414
8,9,Biology,IagYUKyS0cg,Anatomy and Physiology 101: The ULTIMATE Overv...,0.485950,0.154867,0.015322,0.007497,0.367970
9,10,Physics,hyctIDPRSqY,Quantum Physics Full Course | Quantum Mechanic...,0.443990,0.144682,0.063117,0.031150,0.367513



✅ PHASE 9D VALIDATION PASSED

The 1.0000000000000002 value was only floating-point precision and has been safely clipped to 1.0.

Corrected FES file:
/content/revision_phase9d/phase9d_fes_full.csv

Corrected Top-10 file:
/content/revision_phase9d/phase9d_fes_top10.csv


In [45]:
# ============================================================
# PHASE 9E-A — INTRINSIC QUALITY COMPONENTS
# ============================================================
#
# Calculates:
#   1. Semantic Alignment
#   2. Content Depth
#   3. Lexical Diversity
#
# Teaching Score is intentionally handled separately in
# Phase 9E-B because it requires the LLM.
#
# IMPORTANT:
#   - Does NOT modify Phase 1–9D files.
#   - Uses the corrected standard TTR formulation:
#
#       LexicalDiversity = UniqueWords / TotalWords
#
# ============================================================

import os
import re
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import MinMaxScaler


# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

FES_PATH = (
    "/content/revision_phase9d/"
    "phase9d_fes_full.csv"
)

PHASE9A_ROOT = (
    "/content/revision_phase9a"
)

OUTPUT_ROOT = (
    "/content/revision_phase9e"
)

os.makedirs(
    OUTPUT_ROOT,
    exist_ok=True
)


# ------------------------------------------------------------
# 2. LOAD FES DATA
# ------------------------------------------------------------

fes_df = pd.read_csv(
    FES_PATH
)

print(
    f"FES records loaded: {len(fes_df)}"
)

print(
    f"Unique videos: "
    f"{fes_df['video_id'].nunique()}"
)


# ------------------------------------------------------------
# 3. VALIDATE REQUIRED METADATA
# ------------------------------------------------------------

required_columns = [
    "video_id",
    "video_title",
    "description",
    "duration_minutes",
    "domain",
    "query"
]

missing_columns = [
    column
    for column in required_columns
    if column not in fes_df.columns
]

if missing_columns:

    raise ValueError(
        f"Missing required columns: "
        f"{missing_columns}"
    )

print(
    "✅ Required video metadata available."
)


# ------------------------------------------------------------
# 4. LOAD E5 MODEL
# ------------------------------------------------------------

print(
    "\nLoading E5-base-v2..."
)

e5_model = SentenceTransformer(
    "intfloat/e5-base-v2"
)

print(
    "✅ E5-base-v2 loaded."
)


# ------------------------------------------------------------
# 5. BUILD VIDEO CONTENT
# ------------------------------------------------------------
#
# The original pipeline uses title + description for
# intrinsic content evaluation.
#
# We preserve that methodology here.
# ------------------------------------------------------------

fes_df["video_title"] = (
    fes_df["video_title"]
    .fillna("")
    .astype(str)
)

fes_df["description"] = (
    fes_df["description"]
    .fillna("")
    .astype(str)
)

fes_df["content_text"] = (
    fes_df["video_title"]
    + " "
    + fes_df["description"]
)

fes_df["content_text"] = (
    fes_df["content_text"]
    .str.strip()
)


# ------------------------------------------------------------
# 6. CREATE DOMAIN-SPECIFIC SEMANTIC QUERIES
# ------------------------------------------------------------

semantic_queries = {
    "Programming":
        "Complete Python programming course "
        "explaining programming concepts and "
        "practical problem solving",

    "Mathematics":
        "Complete linear algebra course "
        "explaining mathematical concepts, "
        "methods, and problem solving",

    "Physics":
        "Complete quantum mechanics course "
        "explaining physical concepts, "
        "principles, and problem solving",

    "Biology":
        "Complete human anatomy course "
        "explaining anatomical structures, "
        "functions, and biological concepts",

    "Data Science":
        "Complete machine learning course "
        "explaining machine learning concepts, "
        "methods, and practical applications"
}


fes_df["semantic_query"] = (
    fes_df["domain"]
    .map(semantic_queries)
)


# ------------------------------------------------------------
# 7. PREPARE E5 INPUTS
# ------------------------------------------------------------

passage_inputs = [
    "passage: " + text
    for text in fes_df[
        "content_text"
    ].tolist()
]

query_inputs = [
    "query: " + query
    for query in fes_df[
        "semantic_query"
    ].tolist()
]


# ------------------------------------------------------------
# 8. GENERATE CONTENT EMBEDDINGS
# ------------------------------------------------------------

print(
    "\nGenerating video content embeddings..."
)

content_embeddings = e5_model.encode(
    passage_inputs,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print(
    f"Content embedding shape: "
    f"{content_embeddings.shape}"
)


# ------------------------------------------------------------
# 9. GENERATE QUERY EMBEDDINGS
# ------------------------------------------------------------

print(
    "\nGenerating query embeddings..."
)

query_embeddings = e5_model.encode(
    query_inputs,
    batch_size=16,
    show_progress_bar=False,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print(
    f"Query embedding shape: "
    f"{query_embeddings.shape}"
)


# ------------------------------------------------------------
# 10. SEMANTIC ALIGNMENT
# ------------------------------------------------------------
#
# Because both vectors are normalized:
#
#   dot product == cosine similarity
#
# ------------------------------------------------------------

semantic_alignment = np.sum(
    content_embeddings * query_embeddings,
    axis=1
)

fes_df[
    "semantic_alignment"
] = semantic_alignment


# ------------------------------------------------------------
# 11. CONTENT DEPTH
# ------------------------------------------------------------
#
# Duration is used only as a structural proxy for
# content coverage, not as a direct claim that longer
# videos are educationally better.
#
# This distinction is important for the reviewer response.
# ------------------------------------------------------------

duration_values = (
    pd.to_numeric(
        fes_df["duration_minutes"],
        errors="coerce"
    )
    .fillna(0)
)

duration_scaler = MinMaxScaler()

fes_df[
    "depth_score"
] = duration_scaler.fit_transform(
    duration_values.to_numpy().reshape(-1, 1)
).ravel()


# ------------------------------------------------------------
# 12. LEXICAL DIVERSITY — STANDARD TTR
# ------------------------------------------------------------
#
# Standard Type-Token Ratio:
#
#   TTR = UniqueWords / TotalWords
#
# We calculate this from title + description.
# ------------------------------------------------------------

def calculate_lexical_diversity(text):

    text = str(text).lower()

    words = re.findall(
        r"\b[a-z0-9]+\b",
        text
    )

    total_words = len(words)

    if total_words == 0:
        return 0.0

    unique_words = len(
        set(words)
    )

    return (
        unique_words /
        total_words
    )


fes_df[
    "lexical_diversity_raw"
] = (
    fes_df[
        "content_text"
    ]
    .apply(
        calculate_lexical_diversity
    )
)


# ------------------------------------------------------------
# 13. NORMALIZE LEXICAL DIVERSITY
# ------------------------------------------------------------

lexical_scaler = MinMaxScaler()

fes_df[
    "lexical_diversity"
] = lexical_scaler.fit_transform(
    fes_df[
        [
            "lexical_diversity_raw"
        ]
    ]
).ravel()


# ------------------------------------------------------------
# 14. BASIC VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("PHASE 9E-A — COMPONENT SUMMARY")
print("=" * 75)

print(
    f"Videos: {len(fes_df)}"
)

print(
    f"Unique IDs: "
    f"{fes_df['video_id'].nunique()}"
)


print("\nSemantic Alignment:")
print(
    f"  Min    : "
    f"{fes_df['semantic_alignment'].min():.6f}"
)
print(
    f"  Mean   : "
    f"{fes_df['semantic_alignment'].mean():.6f}"
)
print(
    f"  Max    : "
    f"{fes_df['semantic_alignment'].max():.6f}"
)


print("\nDepth Score:")
print(
    f"  Min    : "
    f"{fes_df['depth_score'].min():.6f}"
)
print(
    f"  Mean   : "
    f"{fes_df['depth_score'].mean():.6f}"
)
print(
    f"  Max    : "
    f"{fes_df['depth_score'].max():.6f}"
)


print("\nLexical Diversity — Raw TTR:")
print(
    f"  Min    : "
    f"{fes_df['lexical_diversity_raw'].min():.6f}"
)
print(
    f"  Mean   : "
    f"{fes_df['lexical_diversity_raw'].mean():.6f}"
)
print(
    f"  Max    : "
    f"{fes_df['lexical_diversity_raw'].max():.6f}"
)


print("\nLexical Diversity — Normalized:")
print(
    f"  Min    : "
    f"{fes_df['lexical_diversity'].min():.6f}"
)
print(
    f"  Mean   : "
    f"{fes_df['lexical_diversity'].mean():.6f}"
)
print(
    f"  Max    : "
    f"{fes_df['lexical_diversity'].max():.6f}"
)


# ------------------------------------------------------------
# 15. CHECK INVALID VALUES
# ------------------------------------------------------------

component_columns = [
    "semantic_alignment",
    "depth_score",
    "lexical_diversity_raw",
    "lexical_diversity"
]

invalid_counts = {}

for column in component_columns:

    invalid_counts[column] = int(
        (
            ~np.isfinite(
                fes_df[column]
            )
        ).sum()
    )

print(
    "\nInvalid values:"
)

for column, count in invalid_counts.items():

    print(
        f"  {column}: {count}"
    )


if sum(
    invalid_counts.values()
) > 0:

    raise ValueError(
        "Invalid component values detected."
    )


# ------------------------------------------------------------
# 16. DISPLAY COMPONENTS
# ------------------------------------------------------------

print(
    "\nTop 10 videos by semantic alignment:"
)

display(
    fes_df[
        [
            "domain",
            "video_id",
            "video_title",
            "semantic_alignment",
            "depth_score",
            "lexical_diversity_raw",
            "lexical_diversity"
        ]
    ]
    .sort_values(
        "semantic_alignment",
        ascending=False
    )
    .head(10)
)


# ------------------------------------------------------------
# 17. SAVE PREPARED IQS DATA
# ------------------------------------------------------------

iqs_components_path = os.path.join(
    OUTPUT_ROOT,
    "phase9e_iqs_components.csv"
)

fes_df.to_csv(
    iqs_components_path,
    index=False
)


# ------------------------------------------------------------
# 18. PREPARE TEACHING-SCORE INPUT FILE
# ------------------------------------------------------------
#
# This file will be used in Phase 9E-B for the LLM.
# ------------------------------------------------------------

teaching_input_df = fes_df[
    [
        "video_id",
        "domain",
        "query",
        "video_title",
        "description",
        "duration_minutes",
        "semantic_alignment",
        "depth_score",
        "lexical_diversity_raw"
    ]
].copy()

teaching_input_path = os.path.join(
    OUTPUT_ROOT,
    "phase9e_teaching_score_input.csv"
)

teaching_input_df.to_csv(
    teaching_input_path,
    index=False
)


# ------------------------------------------------------------
# 19. FINAL VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("PHASE 9E-A — FINAL VALIDATION")
print("=" * 75)

validation_passed = True


if len(fes_df) != 99:

    validation_passed = False

    print(
        f"❌ Expected 99 videos, "
        f"found {len(fes_df)}"
    )

else:

    print(
        "✅ 99 videos present"
    )


if fes_df[
    "video_id"
].nunique() != 99:

    validation_passed = False

    print(
        "❌ Video IDs are not unique"
    )

else:

    print(
        "✅ 99 unique video IDs"
    )


for column in [
    "semantic_alignment",
    "depth_score",
    "lexical_diversity_raw",
    "lexical_diversity"
]:

    if not np.isfinite(
        fes_df[column]
    ).all():

        validation_passed = False

        print(
            f"❌ Invalid values in {column}"
        )

    else:

        print(
            f"✅ {column} valid"
        )


print("\n" + "=" * 75)

if validation_passed:

    print(
        "✅ PHASE 9E-A VALIDATION PASSED"
    )

    print(
        "\nThe three deterministic IQS components "
        "are ready for Teaching Score integration."
    )

else:

    print(
        "⚠️ PHASE 9E-A VALIDATION REQUIRES REVIEW"
    )

print("=" * 75)

print(
    f"\nIQS component file:\n"
    f"{iqs_components_path}"
)

print(
    f"\nTeaching-score input:\n"
    f"{teaching_input_path}"
)

FES records loaded: 99
Unique videos: 99
✅ Required video metadata available.

Loading E5-base-v2...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ E5-base-v2 loaded.

Generating video content embeddings...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Content embedding shape: (99, 768)

Generating query embeddings...
Query embedding shape: (99, 768)

PHASE 9E-A — COMPONENT SUMMARY
Videos: 99
Unique IDs: 99

Semantic Alignment:
  Min    : 0.771031
  Mean   : 0.827715
  Max    : 0.866943

Depth Score:
  Min    : 0.000000
  Mean   : 0.142273
  Max    : 1.000000

Lexical Diversity — Raw TTR:
  Min    : 0.311475
  Mean   : 0.582071
  Max    : 1.000000

Lexical Diversity — Normalized:
  Min    : 0.000000
  Mean   : 0.393007
  Max    : 1.000000

Invalid values:
  semantic_alignment: 0
  depth_score: 0
  lexical_diversity_raw: 0
  lexical_diversity: 0

Top 10 videos by semantic alignment:


,domain,video_id,video_title,semantic_alignment,depth_score,lexical_diversity_raw,lexical_diversity
83,Mathematics,JnTa9XtvmfI,Linear Algebra - Full College Course,0.866943,0.647567,0.357367,0.066652
12,Biology,uBGl2BujkPQ,Introduction to Anatomy & Physiology: Crash Co...,0.865320,0.010488,0.598726,0.417197
48,Biology,JNTOahIeCfA,COMPLETE Human Anatomy in 1 Hour! A to Z 3D Hu...,0.863236,0.056328,0.550000,0.346429
8,Biology,IagYUKyS0cg,Anatomy and Physiology 101: The ULTIMATE Overv...,0.859223,0.051593,0.536965,0.327497
42,Biology,CdJgmD2-FWc,INTRO TO HUMAN ANATOMY by PROFESSOR FINK,0.855507,0.054708,0.750000,0.636905
54,Programming,8DvywoWv6fI,Python for Everybody - Full University Python ...,0.854528,0.759004,0.363753,0.075927
19,Data Science,sbmPMtQkjxY,Machine Learning Full Course #Education #Ai #M...,0.854275,0.000093,0.756014,0.645639
9,Physics,hyctIDPRSqY,Quantum Physics Full Course | Quantum Mechanic...,0.853963,0.650343,0.463983,0.221499
86,Mathematics,rSjt1E9WHaQ,Linear Algebra Course – Mathematics for Machin...,0.853142,0.337966,0.586319,0.399178
35,Mathematics,ez8hC76KMJ8,Linear Algebra Books for Self Study,0.851896,0.023753,1.000000,1.000000



PHASE 9E-A — FINAL VALIDATION
✅ 99 videos present
✅ 99 unique video IDs
✅ semantic_alignment valid
✅ depth_score valid
✅ lexical_diversity_raw valid
✅ lexical_diversity valid

✅ PHASE 9E-A VALIDATION PASSED

The three deterministic IQS components are ready for Teaching Score integration.

IQS component file:
/content/revision_phase9e/phase9e_iqs_components.csv

Teaching-score input:
/content/revision_phase9e/phase9e_teaching_score_input.csv


In [47]:
# ============================================================
# PHASE 9E-B — LLM TEACHING SCORE
# ============================================================
#
# Model:
#   gemini-3.5-flash-lite
#
# API:
#   Google Gemini Interactions API
#
# Scoring:
#   0–10
#
# Criteria:
#   1. Clear learning objective
#   2. Structured explanation
#   3. Logical concept progression
#   4. Example usage
#   5. Beginner friendliness
#   6. Practical problem solving
#
# IMPORTANT:
#   - Resumable
#   - Saves after every successful batch
#   - Existing scores are NOT recomputed
#   - Raw LLM responses are retained
#   - Uses 6 videos/request to reduce API calls
#
# ============================================================


from google import genai
from google.colab import userdata

import os
import json
import re
import time
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

INPUT_PATH = (
    "/content/revision_phase9e/"
    "phase9e_teaching_score_input.csv"
)

OUTPUT_ROOT = (
    "/content/revision_phase9e/"
    "teaching_score_results"
)

os.makedirs(
    OUTPUT_ROOT,
    exist_ok=True
)

RESULTS_PATH = os.path.join(
    OUTPUT_ROOT,
    "teaching_scores.csv"
)

RAW_RESPONSES_PATH = os.path.join(
    OUTPUT_ROOT,
    "raw_llm_responses.jsonl"
)


# ------------------------------------------------------------
# 2. LOAD INPUT
# ------------------------------------------------------------

teaching_df = pd.read_csv(
    INPUT_PATH
)

print(
    f"Input records: {len(teaching_df)}"
)

print(
    f"Unique video IDs: "
    f"{teaching_df['video_id'].nunique()}"
)


# ------------------------------------------------------------
# 3. VALIDATE INPUT
# ------------------------------------------------------------

required_columns = [
    "video_id",
    "domain",
    "query",
    "video_title",
    "description",
    "duration_minutes"
]

missing_columns = [
    c
    for c in required_columns
    if c not in teaching_df.columns
]

if missing_columns:

    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

if len(teaching_df) != 99:

    raise ValueError(
        f"Expected 99 videos, found {len(teaching_df)}"
    )

if teaching_df["video_id"].nunique() != 99:

    raise ValueError(
        "Video IDs are not unique."
    )

print(
    "✅ Input validation passed."
)


# ------------------------------------------------------------
# 4. GEMINI CLIENT
# ------------------------------------------------------------

GEMINI_API_KEY = userdata.get(
    "GEMINI_API_KEY"
)

if not GEMINI_API_KEY:

    raise RuntimeError(
        "GEMINI_API_KEY not found in Colab Secrets."
    )

client = genai.Client(
    api_key=GEMINI_API_KEY
)

MODEL_NAME = "gemini-3.5-flash-lite"

print(
    f"Gemini model: {MODEL_NAME}"
)


# ------------------------------------------------------------
# 5. TEACHING-SCORE CRITERIA
# ------------------------------------------------------------

CRITERIA = """
Evaluate the educational teaching quality using these six criteria:

1. Clear learning objective
   - Is it clear what the learner is expected to learn?

2. Structured explanation
   - Is the material organized and presented systematically?

3. Logical concept progression
   - Do concepts appear in a meaningful learning sequence?

4. Example usage
   - Are examples, demonstrations, or concrete applications indicated?

5. Beginner friendliness
   - Is the material accessible and understandable to learners?

6. Practical problem solving
   - Does the content appear to support practical application,
     exercises, projects, or problem solving?
"""


# ------------------------------------------------------------
# 6. PROMPT VERSION
# ------------------------------------------------------------

PROMPT_VERSION = "Phase9E-B_v1"


# ------------------------------------------------------------
# 7. LOAD EXISTING RESULTS IF AVAILABLE
# ------------------------------------------------------------

if os.path.exists(RESULTS_PATH):

    existing_results = pd.read_csv(
        RESULTS_PATH
    )

    print(
        f"Existing Teaching Scores found: "
        f"{len(existing_results)}"
    )

else:

    existing_results = pd.DataFrame(
        columns=[
            "video_id",
            "domain",
            "teaching_score",
            "llm_model",
            "prompt_version",
            "status"
        ]
    )

    print(
        "No previous Teaching Scores found."
    )


# ------------------------------------------------------------
# 8. IDENTIFY COMPLETED VIDEOS
# ------------------------------------------------------------

completed_ids = set()

if not existing_results.empty:

    valid_existing = existing_results[
        existing_results["teaching_score"]
        .notna()
    ]

    completed_ids = set(
        valid_existing[
            "video_id"
        ].astype(str)
    )

print(
    f"Completed videos to skip: "
    f"{len(completed_ids)}"
)


# ------------------------------------------------------------
# 9. HELPER — EXTRACT JSON
# ------------------------------------------------------------

def extract_json(text):

    text = str(text).strip()

    # Remove markdown fences if present
    text = re.sub(
        r"```json\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"```\s*",
        "",
        text
    )

    # Try direct JSON
    try:
        return json.loads(text)

    except Exception:
        pass

    # Try extracting first JSON array
    match = re.search(
        r"\[.*\]",
        text,
        flags=re.DOTALL
    )

    if match:

        try:
            return json.loads(
                match.group(0)
            )

        except Exception:
            pass

    # Try extracting JSON object
    match = re.search(
        r"\{.*\}",
        text,
        flags=re.DOTALL
    )

    if match:

        try:
            return json.loads(
                match.group(0)
            )

        except Exception:
            pass

    return None


# ------------------------------------------------------------
# 10. HELPER — VALIDATE SCORE
# ------------------------------------------------------------

def clean_score(value):

    try:

        score = float(value)

    except Exception:

        return None

    if not np.isfinite(score):

        return None

    # Accept only 0–10
    if score < 0 or score > 10:

        return None

    return round(
        score,
        2
    )


# ------------------------------------------------------------
# 11. BUILD BATCH PROMPT
# ------------------------------------------------------------

def build_batch_prompt(batch_df):

    records = []

    for _, row in batch_df.iterrows():

        description = str(
            row["description"]
        )

        # Keep enough information for evaluation while
        # avoiding unnecessarily huge prompts.
        #
        # Title + first 8000 chars of description.
        description = description[:8000]

        records.append({
            "video_id": str(
                row["video_id"]
            ),
            "domain": str(
                row["domain"]
            ),
            "query": str(
                row["query"]
            ),
            "title": str(
                row["video_title"]
            ),
            "description": description,
            "duration_minutes": float(
                row["duration_minutes"]
            )
        })

    records_json = json.dumps(
        records,
        ensure_ascii=False,
        indent=2
    )

    prompt = f"""
You are an educational-content evaluator.

Your task is to assign ONE overall Teaching Score from 0 to 10
to EACH video provided below.

{CRITERIA}

IMPORTANT EVALUATION RULES:

- Evaluate only the educational information available in the
  provided title and description.
- Do NOT assume that a video contains information that is not
  supported by the provided metadata.
- Do not use popularity, likes, views, comments, or channel
  reputation to determine the Teaching Score.
- Do not reward a video merely because it is long.
- Consider duration only as contextual information.
- Evaluate instructional potential and evidence of educational
  organization from the available metadata.
- Use integer scores from 0 to 10.
- Return exactly one score for every video.
- Preserve every video_id exactly.

Score interpretation:

0–2   = very weak educational evidence
3–4   = limited educational evidence
5–6   = moderate educational quality
7–8   = strong educational quality
9–10  = very strong educational quality

{records_json}

Return ONLY a JSON array in this exact format:

[
  {{
    "video_id": "VIDEO_ID",
    "teaching_score": 8
  }}
]

Do not include explanations outside the JSON array.
"""

    return prompt


# ------------------------------------------------------------
# 12. PREPARE PENDING VIDEOS
# ------------------------------------------------------------

pending_df = teaching_df[
    ~teaching_df[
        "video_id"
    ].astype(str).isin(
        completed_ids
    )
].copy()

print(
    f"\nPending videos: {len(pending_df)}"
)


# ------------------------------------------------------------
# 13. BATCH SIZE
# ------------------------------------------------------------

BATCH_SIZE = 6

total_batches = int(
    np.ceil(
        len(pending_df) /
        BATCH_SIZE
    )
)

print(
    f"Batch size: {BATCH_SIZE}"
)

print(
    f"Gemini requests required: "
    f"{total_batches}"
)


# ------------------------------------------------------------
# 14. PROCESS BATCHES
# ------------------------------------------------------------

new_results = []

for batch_number, start in enumerate(
    range(
        0,
        len(pending_df),
        BATCH_SIZE
    ),
    start=1
):

    batch_df = pending_df.iloc[
        start:start + BATCH_SIZE
    ].copy()

    print("\n" + "=" * 75)
    print(
        f"BATCH {batch_number}/{total_batches}"
    )
    print(
        f"Videos in batch: {len(batch_df)}"
    )
    print("=" * 75)

    print(
        batch_df[
            [
                "video_id",
                "domain",
                "video_title"
            ]
        ].to_string(
            index=False
        )
    )

    prompt = build_batch_prompt(
        batch_df
    )

    try:

        response = client.interactions.create(
            model=MODEL_NAME,
            input=prompt
        )

        raw_output = (
            response.output_text
            if hasattr(
                response,
                "output_text"
            )
            else str(response)
        )

        parsed = extract_json(
            raw_output
        )

        if not isinstance(
            parsed,
            list
        ):

            raise ValueError(
                "Gemini response was not a JSON list."
            )

        # ----------------------------------------------------
        # VALIDATE BATCH RESPONSE
        # ----------------------------------------------------

        expected_ids = set(
            batch_df[
                "video_id"
            ].astype(str)
        )

        response_ids = set()

        batch_results = []

        for item in parsed:

            if not isinstance(
                item,
                dict
            ):
                continue

            video_id = str(
                item.get(
                    "video_id",
                    ""
                )
            )

            score = clean_score(
                item.get(
                    "teaching_score"
                )
            )

            if (
                video_id in expected_ids
                and score is not None
            ):

                response_ids.add(
                    video_id
                )

                original_row = batch_df[
                    batch_df[
                        "video_id"
                    ].astype(str)
                    == video_id
                ].iloc[0]

                batch_results.append({
                    "video_id": video_id,
                    "domain": original_row[
                        "domain"
                    ],
                    "teaching_score": score,
                    "llm_model": MODEL_NAME,
                    "prompt_version": PROMPT_VERSION,
                    "status": "completed"
                })

        missing_ids = (
            expected_ids -
            response_ids
        )

        extra_ids = (
            response_ids -
            expected_ids
        )

        print(
            f"\nValid scores returned: "
            f"{len(batch_results)}/{len(batch_df)}"
        )

        if extra_ids:

            print(
                f"Unexpected IDs: {extra_ids}"
            )

        if missing_ids:

            print(
                f"Missing IDs: {missing_ids}"
            )

            raise ValueError(
                "Batch did not return a valid score "
                "for every requested video."
            )

        # ----------------------------------------------------
        # SAVE RAW RESPONSE
        # ----------------------------------------------------

        raw_record = {
            "batch_number": batch_number,
            "model": MODEL_NAME,
            "prompt_version": PROMPT_VERSION,
            "video_ids": list(
                expected_ids
            ),
            "raw_output": raw_output
        }

        with open(
            RAW_RESPONSES_PATH,
            "a",
            encoding="utf-8"
        ) as f:

            f.write(
                json.dumps(
                    raw_record,
                    ensure_ascii=False
                )
                + "\n"
            )

        # ----------------------------------------------------
        # SAVE SCORES IMMEDIATELY
        # ----------------------------------------------------

        new_results.extend(
            batch_results
        )

        combined_results = pd.concat(
            [
                existing_results,
                pd.DataFrame(new_results)
            ],
            ignore_index=True
        )

        # Remove accidental duplicate IDs
        combined_results = (
            combined_results
            .drop_duplicates(
                subset=["video_id"],
                keep="last"
            )
        )

        combined_results.to_csv(
            RESULTS_PATH,
            index=False
        )

        print(
            f"✅ Batch {batch_number} saved."
        )

        print(
            f"Total saved so far: "
            f"{len(combined_results)}"
        )

        # Small delay between requests
        time.sleep(1)

    except Exception as e:

        print(
            f"\n❌ Batch {batch_number} failed:"
        )

        print(
            type(e).__name__,
            str(e)
        )

        print(
            "\nProgress already saved."
        )

        print(
            "Stop here and inspect the error "
            "before continuing."
        )

        break


# ------------------------------------------------------------
# 15. LOAD FINAL CURRENT RESULTS
# ------------------------------------------------------------

if os.path.exists(
    RESULTS_PATH
):

    final_scores = pd.read_csv(
        RESULTS_PATH
    )

else:

    final_scores = pd.DataFrame()


# ------------------------------------------------------------
# 16. FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("PHASE 9E-B — CURRENT STATUS")
print("=" * 75)

print(
    f"Expected videos : {len(teaching_df)}"
)

print(
    f"Scored videos   : {len(final_scores)}"
)

print(
    f"Missing scores  : "
    f"{len(teaching_df) - len(final_scores)}"
)


if not final_scores.empty:

    print(
        f"\nTeaching Score range: "
        f"{final_scores['teaching_score'].min()} "
        f"→ "
        f"{final_scores['teaching_score'].max()}"
    )

    print(
        f"Mean Teaching Score: "
        f"{final_scores['teaching_score'].mean():.3f}"
    )

    print(
        "\nScore distribution:"
    )

    display(
        final_scores[
            "teaching_score"
        ]
        .value_counts()
        .sort_index()
        .to_frame(
            "count"
        )
    )


if len(final_scores) == len(
    teaching_df
):

    print(
        "\n✅ ALL 99 TEACHING SCORES COMPLETED"
    )

else:

    print(
        "\n⚠️ Teaching Score run is incomplete."
    )

    print(
        "You can safely rerun this cell; "
        "completed video IDs will be skipped."
    )

print(
    f"\nResults file:\n{RESULTS_PATH}"
)

print(
    f"\nRaw responses:\n{RAW_RESPONSES_PATH}"
)

Input records: 99
Unique video IDs: 99
✅ Input validation passed.
Gemini model: gemini-3.5-flash-lite
No previous Teaching Scores found.
Completed videos to skip: 0

Pending videos: 99
Batch size: 6
Gemini requests required: 17

BATCH 1/17
Videos in batch: 6
   video_id      domain                                                       video_title
_uQrJ0TkZlc Programming                                  Python Full Course for Beginners
rfscVS0vtbw Programming               Learn Python - Full Course for Beginners [Tutorial]
XKHEtdqhLK8 Programming                                     Python Full Course for free 🐍
UrsmFxEIp5k Programming Python Tutorial For Beginners in Hindi | Complete Python Course 🔥
kqtD5dpn9C8 Programming         Python for Beginners - Learn Coding with Python in 1 Hour
8Z1C7B98KEY     Biology                   11 Organ Systems of the Human Body (Made Easy!)

Valid scores returned: 6/6
✅ Batch 1 saved.
Total saved so far: 6


/tmp/ipykernel_1521/2686176440.py:683: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_results = pd.concat(



BATCH 2/17
Videos in batch: 6
   video_id      domain                                                                video_title
t8pPdKYpowI Programming      Python Tutorial for Beginners - Learn Python in 5 Hours [FULL COURSE]
ix9cRaBkVe0 Programming                                              Python Full Course for free 🐍
IagYUKyS0cg     Biology Anatomy and Physiology 101: The ULTIMATE Overview (Learn A&P Basics FAST!)
hyctIDPRSqY     Physics                     Quantum Physics Full Course | Quantum Mechanics Course
Rq5gJVxz55Q Programming            Python Full Course for Beginners (13 Hours) – From Zero to Hero
GIJK3dwCWCw     Biology               Immune System, Part 1: Crash Course Anatomy & Physiology #45

Valid scores returned: 6/6
✅ Batch 2 saved.
Total saved so far: 12


/tmp/ipykernel_1521/2686176440.py:683: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_results = pd.concat(



BATCH 3/17
Videos in batch: 6
   video_id       domain                                                                                          video_title
uBGl2BujkPQ      Biology                           Introduction to Anatomy & Physiology: Crash Course Anatomy & Physiology #1
adhgNQmPXZU Data Science                            Machine Learning (தமிழில்) in 25 Minutes:  Absolute Beginner Crash Course
t2_Q2BRzeEE  Programming                                            Python Full Course❤️ | Variables & Data Types | Lecture 1
GwIo3gDZCVQ Data Science Machine Learning Full Course - Learn Machine Learning 10 Hours | Machine Learning Tutorial | Edureka
ow2JE7FTxo0  Mathematics                             Linear Algebra Crash Course | Complete 3-Hour Visual Guide for Beginners
b2q5OFtxm6A Data Science Machine Learning Crash Course-2 Hours | Learn Machine Learning | Machine Learning Tutorial | Edureka

Valid scores returned: 6/6
✅ Batch 3 saved.
Total saved so far: 18


...


BATCH 17/17
Videos in batch: 3
   video_id      domain                                                                                  video_title
eWRfhZUzrAc Programming                                    Python for Beginners – Full Course [Programming Tutorial]
QCPJ0VdpM00 Mathematics                                                          Linear Algebra for Machine Learning
Wzc0rCniHag     Physics Quantum Physics Professor Brutally Honest With Students #viralvideo #viralshorts #shortvideo

Valid scores returned: 3/3
✅ Batch 17 saved.
Total saved so far: 99


/tmp/ipykernel_1521/2686176440.py:683: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_results = pd.concat(



PHASE 9E-B — CURRENT STATUS
Expected videos : 99
Scored videos   : 99
Missing scores  : 0

Teaching Score range: 0.0 → 10.0
Mean Teaching Score: 7.333

Score distribution:


,count
teaching_score,
0.0,1
2.0,4
3.0,6
4.0,8
5.0,5
6.0,2
7.0,10
8.0,18
9.0,33



✅ ALL 99 TEACHING SCORES COMPLETED

Results file:
/content/revision_phase9e/teaching_score_results/teaching_scores.csv

Raw responses:
/content/revision_phase9e/teaching_score_results/raw_llm_responses.jsonl


In [48]:
# ============================================================
# PHASE 9E-C — FINAL INTRINSIC QUALITY SCORE (IQS)
# ============================================================
#
# IQS =
#   0.30 * Semantic Alignment
# + 0.20 * Depth Score
# + 0.20 * Lexical Diversity
# + 0.30 * Normalized Teaching Score
#
# ============================================================

import os
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

COMPONENTS_PATH = (
    "/content/revision_phase9e/"
    "phase9e_iqs_components.csv"
)

TEACHING_PATH = (
    "/content/revision_phase9e/"
    "teaching_score_results/"
    "teaching_scores.csv"
)

OUTPUT_ROOT = (
    "/content/revision_phase9e"
)


# ------------------------------------------------------------
# 2. LOAD IQS COMPONENTS
# ------------------------------------------------------------

components_df = pd.read_csv(
    COMPONENTS_PATH
)

print(
    f"IQS component records: "
    f"{len(components_df)}"
)

print(
    f"Unique component video IDs: "
    f"{components_df['video_id'].nunique()}"
)


# ------------------------------------------------------------
# 3. LOAD TEACHING SCORES
# ------------------------------------------------------------

teaching_df = pd.read_csv(
    TEACHING_PATH
)

print(
    f"\nTeaching Score records: "
    f"{len(teaching_df)}"
)

print(
    f"Unique Teaching Score IDs: "
    f"{teaching_df['video_id'].nunique()}"
)


# ------------------------------------------------------------
# 4. VALIDATE INPUT COHORTS
# ------------------------------------------------------------

component_ids = set(
    components_df[
        "video_id"
    ].astype(str)
)

teaching_ids = set(
    teaching_df[
        "video_id"
    ].astype(str)
)


print("\nCohort comparison:")

print(
    f"Components ∩ Teaching: "
    f"{len(component_ids & teaching_ids)}"
)

print(
    f"Components only: "
    f"{len(component_ids - teaching_ids)}"
)

print(
    f"Teaching only: "
    f"{len(teaching_ids - component_ids)}"
)


if component_ids != teaching_ids:

    raise ValueError(
        "IQS component and Teaching Score "
        "cohorts do not match."
    )

print(
    "✅ Exact cohort match."
)


# ------------------------------------------------------------
# 5. MERGE
# ------------------------------------------------------------

iqs_df = components_df.merge(
    teaching_df[
        [
            "video_id",
            "teaching_score",
            "llm_model",
            "prompt_version",
            "status"
        ]
    ],
    on="video_id",
    how="inner"
)


print(
    f"\nMerged IQS records: "
    f"{len(iqs_df)}"
)

print(
    f"Unique merged IDs: "
    f"{iqs_df['video_id'].nunique()}"
)


# ------------------------------------------------------------
# 6. TEACHING SCORE NORMALIZATION
# ------------------------------------------------------------

iqs_df[
    "teaching_score_normalized"
] = (
    iqs_df[
        "teaching_score"
    ]
    / 10.0
)


# ------------------------------------------------------------
# 7. FINAL IQS
# ------------------------------------------------------------

iqs_df[
    "intrinsic_quality_score"
] = (
    0.30
    * iqs_df[
        "semantic_alignment"
    ]
    +
    0.20
    * iqs_df[
        "depth_score"
    ]
    +
    0.20
    * iqs_df[
        "lexical_diversity"
    ]
    +
    0.30
    * iqs_df[
        "teaching_score_normalized"
    ]
)


# ------------------------------------------------------------
# 8. GLOBAL IQS RANK
# ------------------------------------------------------------

iqs_df = iqs_df.sort_values(
    "intrinsic_quality_score",
    ascending=False
).reset_index(
    drop=True
)

iqs_df[
    "global_iqs_rank"
] = np.arange(
    1,
    len(iqs_df) + 1
)


# ------------------------------------------------------------
# 9. DOMAIN-WISE IQS RANK
# ------------------------------------------------------------

iqs_df[
    "domain_iqs_rank"
] = (
    iqs_df
    .groupby("domain")[
        "intrinsic_quality_score"
    ]
    .rank(
        method="first",
        ascending=False
    )
    .astype(int)
)


# ------------------------------------------------------------
# 10. COMPONENT CONTRIBUTIONS
# ------------------------------------------------------------

iqs_df[
    "semantic_contribution"
] = (
    0.30
    * iqs_df[
        "semantic_alignment"
    ]
)

iqs_df[
    "depth_contribution"
] = (
    0.20
    * iqs_df[
        "depth_score"
    ]
)

iqs_df[
    "lexical_contribution"
] = (
    0.20
    * iqs_df[
        "lexical_diversity"
    ]
)

iqs_df[
    "teaching_contribution"
] = (
    0.30
    * iqs_df[
        "teaching_score_normalized"
    ]
)


# ------------------------------------------------------------
# 11. DOMINANT CONTRIBUTION
# ------------------------------------------------------------

contribution_columns = [
    "semantic_contribution",
    "depth_contribution",
    "lexical_contribution",
    "teaching_contribution"
]

component_names = {
    "semantic_contribution":
        "Semantic Alignment",

    "depth_contribution":
        "Depth",

    "lexical_contribution":
        "Lexical Diversity",

    "teaching_contribution":
        "Teaching Score"
}


iqs_df[
    "dominant_iqs_component"
] = (
    iqs_df[
        contribution_columns
    ]
    .idxmax(axis=1)
    .map(component_names)
)


# ------------------------------------------------------------
# 12. SAVE COMPLETE IQS
# ------------------------------------------------------------

full_output = os.path.join(
    OUTPUT_ROOT,
    "phase9e_iqs_full.csv"
)

iqs_df.to_csv(
    full_output,
    index=False
)


# ------------------------------------------------------------
# 13. SAVE TOP 10
# ------------------------------------------------------------

top10_output = os.path.join(
    OUTPUT_ROOT,
    "phase9e_iqs_top10.csv"
)

iqs_df.head(10).to_csv(
    top10_output,
    index=False
)


# ------------------------------------------------------------
# 14. DOMAIN SUMMARY
# ------------------------------------------------------------

domain_summary = (
    iqs_df
    .groupby("domain")
    .agg(
        videos=(
            "video_id",
            "nunique"
        ),
        mean_iqs=(
            "intrinsic_quality_score",
            "mean"
        ),
        std_iqs=(
            "intrinsic_quality_score",
            "std"
        ),
        min_iqs=(
            "intrinsic_quality_score",
            "min"
        ),
        max_iqs=(
            "intrinsic_quality_score",
            "max"
        ),
        mean_semantic_alignment=(
            "semantic_alignment",
            "mean"
        ),
        mean_depth=(
            "depth_score",
            "mean"
        ),
        mean_lexical_diversity=(
            "lexical_diversity",
            "mean"
        ),
        mean_teaching_score=(
            "teaching_score",
            "mean"
        )
    )
    .reset_index()
)


domain_summary_output = os.path.join(
    OUTPUT_ROOT,
    "phase9e_iqs_domain_summary.csv"
)

domain_summary.to_csv(
    domain_summary_output,
    index=False
)


# ------------------------------------------------------------
# 15. TOP 10 DISPLAY
# ------------------------------------------------------------

print("\n" + "=" * 85)
print("TOP 10 VIDEOS BY INTRINSIC QUALITY SCORE")
print("=" * 85)

display(
    iqs_df[
        [
            "global_iqs_rank",
            "domain",
            "video_id",
            "video_title",
            "semantic_alignment",
            "depth_score",
            "lexical_diversity",
            "teaching_score",
            "intrinsic_quality_score",
            "dominant_iqs_component"
        ]
    ].head(10)
)


# ------------------------------------------------------------
# 16. DOMAIN SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 85)
print("DOMAIN-WISE IQS SUMMARY")
print("=" * 85)

display(
    domain_summary
)


# ------------------------------------------------------------
# 17. TEACHING SCORE SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 85)
print("TEACHING SCORE SUMMARY")
print("=" * 85)

print(
    f"Mean: "
    f"{iqs_df['teaching_score'].mean():.3f}"
)

print(
    f"Std: "
    f"{iqs_df['teaching_score'].std():.3f}"
)

print(
    f"Min: "
    f"{iqs_df['teaching_score'].min():.1f}"
)

print(
    f"Max: "
    f"{iqs_df['teaching_score'].max():.1f}"
)


# ------------------------------------------------------------
# 18. VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 85)
print("PHASE 9E-C — FINAL VALIDATION")
print("=" * 85)

validation_passed = True


# Record count
if len(iqs_df) != 99:

    validation_passed = False

    print(
        f"❌ Expected 99 videos, "
        f"found {len(iqs_df)}"
    )

else:

    print(
        "✅ 99 video records"
    )


# Unique IDs
if iqs_df[
    "video_id"
].nunique() != 99:

    validation_passed = False

    print(
        "❌ Duplicate video IDs"
    )

else:

    print(
        "✅ 99 unique video IDs"
    )


# Teaching score validity
if (
    iqs_df["teaching_score"]
    .min() < 0
    or
    iqs_df["teaching_score"]
    .max() > 10
):

    validation_passed = False

    print(
        "❌ Teaching Score outside [0,10]"
    )

else:

    print(
        "✅ Teaching Scores within [0,10]"
    )


# Teaching normalized
if not (
    iqs_df[
        "teaching_score_normalized"
    ].between(
        0,
        1
    ).all()
):

    validation_passed = False

    print(
        "❌ Normalized Teaching Score invalid"
    )

else:

    print(
        "✅ Normalized Teaching Score valid"
    )


# IQS finite
if not np.isfinite(
    iqs_df[
        "intrinsic_quality_score"
    ]
).all():

    validation_passed = False

    print(
        "❌ Non-finite IQS values"
    )

else:

    print(
        "✅ All IQS values finite"
    )


# IQS missing
if iqs_df[
    "intrinsic_quality_score"
].isna().sum() > 0:

    validation_passed = False

    print(
        "❌ Missing IQS values"
    )

else:

    print(
        "✅ No missing IQS values"
    )


# ------------------------------------------------------------
# 19. FINAL RESULT
# ------------------------------------------------------------

print("\n" + "=" * 85)

if validation_passed:

    print(
        "✅ PHASE 9E-C VALIDATION PASSED"
    )

    print(
        "Intrinsic Quality Score successfully "
        "computed for all 99 videos."
    )

else:

    print(
        "⚠️ PHASE 9E-C VALIDATION REQUIRES REVIEW"
    )

print("=" * 85)


print(
    f"\nFull IQS file:\n{full_output}"
)

print(
    f"\nTop-10 IQS file:\n{top10_output}"
)

print(
    f"\nDomain summary:\n{domain_summary_output}"
)

IQS component records: 99
Unique component video IDs: 99

Teaching Score records: 99
Unique Teaching Score IDs: 99

Cohort comparison:
Components ∩ Teaching: 99
Components only: 0
Teaching only: 0
✅ Exact cohort match.

Merged IQS records: 99
Unique merged IDs: 99

TOP 10 VIDEOS BY INTRINSIC QUALITY SCORE


,global_iqs_rank,domain,video_id,video_title,semantic_alignment,depth_score,lexical_diversity,teaching_score,intrinsic_quality_score,dominant_iqs_component
0,1,Data Science,hDKCxebp88A,Machine Learning with Python and Scikit-Learn ...,0.831077,1.000000,0.246557,10.0,0.798634,Teaching Score
1,2,Programming,nLRL_NcnK-4,Harvard CS50’s Introduction to Programming wit...,0.838486,0.886373,0.234522,10.0,0.775725,Teaching Score
2,3,Data Science,NWONeJKn6kc,Machine Learning Course for Beginners,0.842298,0.548145,0.457412,10.0,0.753801,Teaching Score
3,4,Mathematics,QCPJ0VdpM00,Linear Algebra for Machine Learning,0.832583,0.600339,0.365862,10.0,0.743015,Teaching Score
4,5,Programming,8DvywoWv6fI,Python for Everybody - Full University Python ...,0.854528,0.759004,0.075927,10.0,0.723344,Teaching Score
5,6,Programming,Rq5gJVxz55Q,Python Full Course for Beginners (13 Hours) – ...,0.832468,0.724501,0.222849,9.0,0.709210,Teaching Score
6,7,Mathematics,rSjt1E9WHaQ,Linear Algebra Course – Mathematics for Machin...,0.853142,0.337966,0.399178,10.0,0.703371,Teaching Score
7,8,Mathematics,JnTa9XtvmfI,Linear Algebra - Full College Course,0.866943,0.647567,0.066652,10.0,0.702927,Teaching Score
8,9,Programming,H2EJuAcrZYU,Python Full Course for Beginners | Complete Al...,0.843421,0.480234,0.263063,10.0,0.701686,Teaching Score
9,10,Physics,hyctIDPRSqY,Quantum Physics Full Course | Quantum Mechanic...,0.853963,0.650343,0.221499,9.0,0.700557,Teaching Score



DOMAIN-WISE IQS SUMMARY


,domain,videos,mean_iqs,std_iqs,min_iqs,max_iqs,mean_semantic_alignment,mean_depth,mean_lexical_diversity,mean_teaching_score
0,Biology,20,0.564768,0.060410,0.430106,0.634975,0.826122,0.019002,0.433156,7.550000
1,Data Science,20,0.569459,0.097000,0.428161,0.798634,0.833870,0.175551,0.325938,7.300000
2,Mathematics,20,0.582845,0.102178,0.368486,0.743015,0.835656,0.135861,0.399880,7.500000
3,Physics,19,0.545278,0.095730,0.310845,0.700557,0.812773,0.061065,0.459321,6.578947
4,Programming,20,0.612999,0.118245,0.353152,0.775725,0.829406,0.315826,0.350058,7.700000



TEACHING SCORE SUMMARY
Mean: 7.333
Std: 2.445
Min: 0.0
Max: 10.0

PHASE 9E-C — FINAL VALIDATION
✅ 99 video records
✅ 99 unique video IDs
✅ Teaching Scores within [0,10]
✅ Normalized Teaching Score valid
✅ All IQS values finite
✅ No missing IQS values

✅ PHASE 9E-C VALIDATION PASSED
Intrinsic Quality Score successfully computed for all 99 videos.

Full IQS file:
/content/revision_phase9e/phase9e_iqs_full.csv

Top-10 IQS file:
/content/revision_phase9e/phase9e_iqs_top10.csv

Domain summary:
/content/revision_phase9e/phase9e_iqs_domain_summary.csv


In [49]:
# ============================================================
# PHASE 9F — ADAPTIVE HYBRID RANKING
# ============================================================
#
# Hybrid Score:
#
#   Hybrid = alpha * FES + (1-alpha) * IQS
#
# where:
#
#   EngagementStrength = log1p(relevant_comment_count)
#
#   alpha =
#       EngagementStrength /
#       (EngagementStrength + k)
#
#   k = 5
#
# This phase combines:
#   Phase 9D → Final Engagement Score (FES)
#   Phase 9E → Intrinsic Quality Score (IQS)
#
# ============================================================


import os
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

FES_PATH = (
    "/content/revision_phase9d/"
    "phase9d_fes_full.csv"
)

IQS_PATH = (
    "/content/revision_phase9e/"
    "phase9e_iqs_full.csv"
)

OUTPUT_ROOT = (
    "/content/revision_phase9f"
)

os.makedirs(
    OUTPUT_ROOT,
    exist_ok=True
)


# ------------------------------------------------------------
# 2. LOAD FES
# ------------------------------------------------------------

fes_df = pd.read_csv(
    FES_PATH
)

print(
    f"FES records: {len(fes_df)}"
)

print(
    f"FES unique videos: "
    f"{fes_df['video_id'].nunique()}"
)


# ------------------------------------------------------------
# 3. LOAD IQS
# ------------------------------------------------------------

iqs_df = pd.read_csv(
    IQS_PATH
)

print(
    f"\nIQS records: {len(iqs_df)}"
)

print(
    f"IQS unique videos: "
    f"{iqs_df['video_id'].nunique()}"
)


# ------------------------------------------------------------
# 4. VALIDATE INPUT COHORTS
# ------------------------------------------------------------

fes_ids = set(
    fes_df[
        "video_id"
    ].astype(str)
)

iqs_ids = set(
    iqs_df[
        "video_id"
    ].astype(str)
)


print("\n" + "=" * 75)
print("COHORT VALIDATION")
print("=" * 75)

print(
    f"FES ∩ IQS       : "
    f"{len(fes_ids & iqs_ids)}"
)

print(
    f"FES only        : "
    f"{len(fes_ids - iqs_ids)}"
)

print(
    f"IQS only        : "
    f"{len(iqs_ids - fes_ids)}"
)


if fes_ids != iqs_ids:

    raise ValueError(
        "FES and IQS cohorts do not match."
    )

print(
    "✅ Exact 99-video cohort match."
)


# ------------------------------------------------------------
# 5. SELECT REQUIRED FES COLUMNS
# ------------------------------------------------------------

fes_required = [
    "video_id",
    "domain",
    "query",
    "video_title",
    "final_engagement_score",
    "relevant_comment_count"
]

missing_fes = [
    col
    for col in fes_required
    if col not in fes_df.columns
]

if missing_fes:

    raise ValueError(
        f"Missing FES columns: {missing_fes}"
    )


fes_input = fes_df[
    fes_required
].copy()


# ------------------------------------------------------------
# 6. SELECT REQUIRED IQS COLUMNS
# ------------------------------------------------------------

iqs_required = [
    "video_id",
    "intrinsic_quality_score"
]

missing_iqs = [
    col
    for col in iqs_required
    if col not in iqs_df.columns
]

if missing_iqs:

    raise ValueError(
        f"Missing IQS columns: {missing_iqs}"
    )


iqs_input = iqs_df[
    iqs_required
].copy()


# ------------------------------------------------------------
# 7. MERGE FES + IQS
# ------------------------------------------------------------

hybrid_df = fes_input.merge(
    iqs_input,
    on="video_id",
    how="inner"
)


print(
    f"\nMerged records: "
    f"{len(hybrid_df)}"
)

print(
    f"Merged unique videos: "
    f"{hybrid_df['video_id'].nunique()}"
)


# ------------------------------------------------------------
# 8. VALIDATE MERGE
# ------------------------------------------------------------

if len(hybrid_df) != 99:

    raise ValueError(
        f"Expected 99 merged videos, "
        f"found {len(hybrid_df)}"
    )

if (
    hybrid_df["video_id"].nunique()
    != 99
):

    raise ValueError(
        "Duplicate video IDs after merge."
    )

print(
    "✅ Hybrid cohort validation passed."
)


# ------------------------------------------------------------
# 9. NUMERIC CONVERSION
# ------------------------------------------------------------

hybrid_df[
    "final_engagement_score"
] = pd.to_numeric(
    hybrid_df[
        "final_engagement_score"
    ],
    errors="coerce"
)

hybrid_df[
    "intrinsic_quality_score"
] = pd.to_numeric(
    hybrid_df[
        "intrinsic_quality_score"
    ],
    errors="coerce"
)

hybrid_df[
    "relevant_comment_count"
] = pd.to_numeric(
    hybrid_df[
        "relevant_comment_count"
    ],
    errors="coerce"
)


# ------------------------------------------------------------
# 10. CHECK MISSING VALUES
# ------------------------------------------------------------

required_numeric = [
    "final_engagement_score",
    "intrinsic_quality_score",
    "relevant_comment_count"
]

missing_counts = (
    hybrid_df[
        required_numeric
    ]
    .isna()
    .sum()
)

print("\nMissing values:")

display(
    missing_counts.to_frame(
        "missing_count"
    )
)

if missing_counts.sum() > 0:

    raise ValueError(
        "Missing numeric values detected."
    )


# ------------------------------------------------------------
# 11. ENGAGEMENT STRENGTH
# ------------------------------------------------------------
#
# log1p(x) = log(1+x)
#
# This compresses the effect of very large
# comment counts while preserving ordering.
# ------------------------------------------------------------

hybrid_df[
    "engagement_strength"
] = np.log1p(
    hybrid_df[
        "relevant_comment_count"
    ]
)


# ------------------------------------------------------------
# 12. ADAPTIVE ALPHA
# ------------------------------------------------------------

K = 5.0

hybrid_df[
    "alpha"
] = (
    hybrid_df[
        "engagement_strength"
    ]
    /
    (
        hybrid_df[
            "engagement_strength"
        ]
        + K
    )
)


# ------------------------------------------------------------
# 13. IQS WEIGHT
# ------------------------------------------------------------

hybrid_df[
    "iqs_weight"
] = (
    1.0
    -
    hybrid_df[
        "alpha"
    ]
)


# ------------------------------------------------------------
# 14. COMPONENT CONTRIBUTIONS
# ------------------------------------------------------------

hybrid_df[
    "fes_contribution"
] = (
    hybrid_df[
        "alpha"
    ]
    *
    hybrid_df[
        "final_engagement_score"
    ]
)

hybrid_df[
    "iqs_contribution"
] = (
    hybrid_df[
        "iqs_weight"
    ]
    *
    hybrid_df[
        "intrinsic_quality_score"
    ]
)


# ------------------------------------------------------------
# 15. HYBRID SCORE
# ------------------------------------------------------------

hybrid_df[
    "hybrid_score"
] = (
    hybrid_df[
        "fes_contribution"
    ]
    +
    hybrid_df[
        "iqs_contribution"
    ]
)


# ------------------------------------------------------------
# 16. DOMINANT CONTRIBUTION
# ------------------------------------------------------------

hybrid_df[
    "dominant_component"
] = np.where(
    hybrid_df[
        "fes_contribution"
    ]
    >
    hybrid_df[
        "iqs_contribution"
    ],
    "FES",
    "IQS"
)


# ------------------------------------------------------------
# 17. GLOBAL FES RANK
# ------------------------------------------------------------

hybrid_df[
    "fes_rank"
] = (
    hybrid_df[
        "final_engagement_score"
    ]
    .rank(
        method="first",
        ascending=False
    )
    .astype(int)
)


# ------------------------------------------------------------
# 18. GLOBAL IQS RANK
# ------------------------------------------------------------

hybrid_df[
    "iqs_rank"
] = (
    hybrid_df[
        "intrinsic_quality_score"
    ]
    .rank(
        method="first",
        ascending=False
    )
    .astype(int)
)


# ------------------------------------------------------------
# 19. GLOBAL HYBRID RANK
# ------------------------------------------------------------

hybrid_df = hybrid_df.sort_values(
    "hybrid_score",
    ascending=False
).reset_index(
    drop=True
)

hybrid_df[
    "hybrid_rank"
] = np.arange(
    1,
    len(hybrid_df) + 1
)


# ------------------------------------------------------------
# 20. RANK MOVEMENT
# ------------------------------------------------------------
#
# Positive value:
#   video moved upward relative to source ranking
#
# Negative value:
#   video moved downward
#
# We calculate movement from both FES and IQS.
# ------------------------------------------------------------

hybrid_df[
    "movement_from_fes"
] = (
    hybrid_df[
        "fes_rank"
    ]
    -
    hybrid_df[
        "hybrid_rank"
    ]
)

hybrid_df[
    "movement_from_iqs"
] = (
    hybrid_df[
        "iqs_rank"
    ]
    -
    hybrid_df[
        "hybrid_rank"
    ]
)


# ------------------------------------------------------------
# 21. DOMAIN-WISE HYBRID RANK
# ------------------------------------------------------------

hybrid_df[
    "domain_hybrid_rank"
] = (
    hybrid_df
    .groupby("domain")[
        "hybrid_score"
    ]
    .rank(
        method="first",
        ascending=False
    )
    .astype(int)
)


# ------------------------------------------------------------
# 22. DOMAIN-WISE FES / IQS RANK
# ------------------------------------------------------------

hybrid_df[
    "domain_fes_rank"
] = (
    hybrid_df
    .groupby("domain")[
        "final_engagement_score"
    ]
    .rank(
        method="first",
        ascending=False
    )
    .astype(int)
)

hybrid_df[
    "domain_iqs_rank"
] = (
    hybrid_df
    .groupby("domain")[
        "intrinsic_quality_score"
    ]
    .rank(
        method="first",
        ascending=False
    )
    .astype(int)
)


# ------------------------------------------------------------
# 23. SAVE COMPLETE HYBRID DATASET
# ------------------------------------------------------------

full_output = os.path.join(
    OUTPUT_ROOT,
    "phase9f_hybrid_full.csv"
)

hybrid_df.to_csv(
    full_output,
    index=False
)


# ------------------------------------------------------------
# 24. SAVE TOP 10
# ------------------------------------------------------------

top10_output = os.path.join(
    OUTPUT_ROOT,
    "phase9f_hybrid_top10.csv"
)

hybrid_df.head(10).to_csv(
    top10_output,
    index=False
)


# ------------------------------------------------------------
# 25. SAVE TOP 20
# ------------------------------------------------------------

top20_output = os.path.join(
    OUTPUT_ROOT,
    "phase9f_hybrid_top20.csv"
)

hybrid_df.head(20).to_csv(
    top20_output,
    index=False
)


# ------------------------------------------------------------
# 26. DOMAIN SUMMARY
# ------------------------------------------------------------

domain_summary = (
    hybrid_df
    .groupby("domain")
    .agg(
        videos=(
            "video_id",
            "nunique"
        ),

        mean_fes=(
            "final_engagement_score",
            "mean"
        ),

        mean_iqs=(
            "intrinsic_quality_score",
            "mean"
        ),

        mean_hybrid=(
            "hybrid_score",
            "mean"
        ),

        std_hybrid=(
            "hybrid_score",
            "std"
        ),

        min_hybrid=(
            "hybrid_score",
            "min"
        ),

        max_hybrid=(
            "hybrid_score",
            "max"
        ),

        mean_alpha=(
            "alpha",
            "mean"
        ),

        mean_engagement_strength=(
            "engagement_strength",
            "mean"
        ),

        fes_dominant_videos=(
            "dominant_component",
            lambda x:
                (x == "FES").sum()
        ),

        iqs_dominant_videos=(
            "dominant_component",
            lambda x:
                (x == "IQS").sum()
        )
    )
    .reset_index()
)


domain_summary_output = os.path.join(
    OUTPUT_ROOT,
    "phase9f_domain_summary.csv"
)

domain_summary.to_csv(
    domain_summary_output,
    index=False
)


# ------------------------------------------------------------
# 27. ALPHA SUMMARY
# ------------------------------------------------------------

alpha_summary = pd.DataFrame({
    "statistic": [
        "count",
        "mean",
        "std",
        "min",
        "median",
        "max"
    ],

    "alpha": [
        hybrid_df["alpha"].count(),
        hybrid_df["alpha"].mean(),
        hybrid_df["alpha"].std(),
        hybrid_df["alpha"].min(),
        hybrid_df["alpha"].median(),
        hybrid_df["alpha"].max()
    ]
})


alpha_output = os.path.join(
    OUTPUT_ROOT,
    "phase9f_alpha_summary.csv"
)

alpha_summary.to_csv(
    alpha_output,
    index=False
)


# ------------------------------------------------------------
# 28. TOP 10 HYBRID RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 110)
print("TOP 10 VIDEOS — ADAPTIVE HYBRID RANKING")
print("=" * 110)

display(
    hybrid_df[
        [
            "hybrid_rank",
            "domain",
            "video_id",
            "video_title",
            "final_engagement_score",
            "intrinsic_quality_score",
            "relevant_comment_count",
            "engagement_strength",
            "alpha",
            "fes_contribution",
            "iqs_contribution",
            "hybrid_score",
            "dominant_component"
        ]
    ].head(10)
)


# ------------------------------------------------------------
# 29. DOMAIN SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 110)
print("DOMAIN-WISE HYBRID SUMMARY")
print("=" * 110)

display(
    domain_summary
)


# ------------------------------------------------------------
# 30. ALPHA SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 110)
print("ADAPTIVE ALPHA SUMMARY")
print("=" * 110)

display(
    alpha_summary
)


# ------------------------------------------------------------
# 31. RANK MOVEMENT — TOP MOVERS UP
# ------------------------------------------------------------

print("\n" + "=" * 110)
print("TOP 10 VIDEOS MOVING UP FROM FES RANK")
print("=" * 110)

display(
    hybrid_df[
        [
            "hybrid_rank",
            "fes_rank",
            "movement_from_fes",
            "domain",
            "video_id",
            "video_title",
            "final_engagement_score",
            "intrinsic_quality_score",
            "hybrid_score"
        ]
    ]
    .sort_values(
        "movement_from_fes",
        ascending=False
    )
    .head(10)
)


# ------------------------------------------------------------
# 32. RANK MOVEMENT — TOP MOVERS DOWN
# ------------------------------------------------------------

print("\n" + "=" * 110)
print("TOP 10 VIDEOS MOVING DOWN FROM FES RANK")
print("=" * 110)

display(
    hybrid_df[
        [
            "hybrid_rank",
            "fes_rank",
            "movement_from_fes",
            "domain",
            "video_id",
            "video_title",
            "final_engagement_score",
            "intrinsic_quality_score",
            "hybrid_score"
        ]
    ]
    .sort_values(
        "movement_from_fes",
        ascending=True
    )
    .head(10)
)


# ------------------------------------------------------------
# 33. VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 110)
print("PHASE 9F — FINAL VALIDATION")
print("=" * 110)

validation_passed = True


# Record count
if len(hybrid_df) != 99:

    validation_passed = False

    print(
        f"❌ Expected 99 records, "
        f"found {len(hybrid_df)}"
    )

else:

    print(
        "✅ 99 hybrid records"
    )


# Unique IDs
if (
    hybrid_df["video_id"].nunique()
    != 99
):

    validation_passed = False

    print(
        "❌ Duplicate video IDs"
    )

else:

    print(
        "✅ 99 unique video IDs"
    )


# Alpha range
if not hybrid_df[
    "alpha"
].between(
    0,
    1
).all():

    validation_passed = False

    print(
        "❌ Alpha outside [0,1]"
    )

else:

    print(
        "✅ Alpha values within [0,1]"
    )


# IQS weight range
if not hybrid_df[
    "iqs_weight"
].between(
    0,
    1
).all():

    validation_passed = False

    print(
        "❌ IQS weights outside [0,1]"
    )

else:

    print(
        "✅ IQS weights within [0,1]"
    )


# Hybrid finite
if not np.isfinite(
    hybrid_df[
        "hybrid_score"
    ]
).all():

    validation_passed = False

    print(
        "❌ Non-finite Hybrid scores"
    )

else:

    print(
        "✅ All Hybrid scores finite"
    )


# Hybrid missing
if hybrid_df[
    "hybrid_score"
].isna().sum() > 0:

    validation_passed = False

    print(
        "❌ Missing Hybrid scores"
    )

else:

    print(
        "✅ No missing Hybrid scores"
    )


# Rank integrity
if set(
    hybrid_df["hybrid_rank"]
) != set(
    range(
        1,
        100
    )
):

    validation_passed = False

    print(
        "❌ Hybrid ranking is not 1–99"
    )

else:

    print(
        "✅ Hybrid ranking is complete 1–99"
    )


# Contribution identity
contribution_error = np.max(
    np.abs(
        hybrid_df[
            "hybrid_score"
        ]
        -
        (
            hybrid_df[
                "fes_contribution"
            ]
            +
            hybrid_df[
                "iqs_contribution"
            ]
        )
    )
)

print(
    f"\nMaximum contribution error: "
    f"{contribution_error:.12f}"
)

if contribution_error > 1e-10:

    validation_passed = False

    print(
        "❌ Contribution identity failed"
    )

else:

    print(
        "✅ Hybrid = FES contribution + "
        "IQS contribution"
    )


# Alpha formula validation
expected_alpha = (
    hybrid_df[
        "engagement_strength"
    ]
    /
    (
        hybrid_df[
            "engagement_strength"
        ]
        + K
    )
)

alpha_error = np.max(
    np.abs(
        hybrid_df[
            "alpha"
        ]
        -
        expected_alpha
    )
)

print(
    f"Maximum alpha error: "
    f"{alpha_error:.12f}"
)

if alpha_error > 1e-10:

    validation_passed = False

    print(
        "❌ Alpha formula validation failed"
    )

else:

    print(
        "✅ Alpha formula validated"
    )


# ------------------------------------------------------------
# 34. FINAL RESULT
# ------------------------------------------------------------

print("\n" + "=" * 110)

if validation_passed:

    print(
        "✅ PHASE 9F VALIDATION PASSED"
    )

    print(
        "Adaptive Hybrid Ranking successfully "
        "computed for all 99 videos."
    )

else:

    print(
        "⚠️ PHASE 9F VALIDATION REQUIRES REVIEW"
    )

print("=" * 110)


# ------------------------------------------------------------
# 35. OUTPUT PATHS
# ------------------------------------------------------------

print(
    f"\nFull Hybrid file:\n{full_output}"
)

print(
    f"\nTop-10 file:\n{top10_output}"
)

print(
    f"\nTop-20 file:\n{top20_output}"
)

print(
    f"\nDomain summary:\n{domain_summary_output}"
)

print(
    f"\nAlpha summary:\n{alpha_output}"
)

FES records: 99
FES unique videos: 99

IQS records: 99
IQS unique videos: 99

COHORT VALIDATION
FES ∩ IQS       : 99
FES only        : 0
IQS only        : 0
✅ Exact 99-video cohort match.

Merged records: 99
Merged unique videos: 99
✅ Hybrid cohort validation passed.

Missing values:


,missing_count
final_engagement_score,0
intrinsic_quality_score,0
relevant_comment_count,0



TOP 10 VIDEOS — ADAPTIVE HYBRID RANKING


,hybrid_rank,domain,video_id,video_title,final_engagement_score,intrinsic_quality_score,relevant_comment_count,engagement_strength,alpha,fes_contribution,iqs_contribution,hybrid_score,dominant_component
0,1,Programming,_uQrJ0TkZlc,Python Full Course for Beginners,0.714660,0.643385,40,3.713572,0.426183,0.304576,0.369186,0.673762,IQS
1,2,Programming,rfscVS0vtbw,Learn Python - Full Course for Beginners [Tuto...,0.670344,0.647528,53,3.988984,0.443764,0.297474,0.360179,0.657653,IQS
2,3,Programming,XKHEtdqhLK8,Python Full Course for free 🐍,0.556236,0.696523,40,3.713572,0.426183,0.237058,0.399677,0.636735,IQS
3,4,Programming,UrsmFxEIp5k,Python Tutorial For Beginners in Hindi | Compl...,0.555875,0.659890,63,4.158883,0.454082,0.252413,0.360246,0.612658,IQS
4,5,Mathematics,ow2JE7FTxo0,Linear Algebra Crash Course | Complete 3-Hour ...,0.348359,0.637839,2,1.098612,0.180141,0.062754,0.522937,0.585691,IQS
5,6,Programming,nLRL_NcnK-4,Harvard CS50’s Introduction to Programming wit...,0.330222,0.775725,48,3.891820,0.437685,0.144533,0.436201,0.580734,IQS
6,7,Programming,ix9cRaBkVe0,Python Full Course for free 🐍,0.379414,0.697736,37,3.637586,0.421135,0.159784,0.403895,0.563679,IQS
7,8,Physics,hyctIDPRSqY,Quantum Physics Full Course | Quantum Mechanic...,0.367513,0.700557,33,3.526361,0.413583,0.151997,0.410819,0.562816,IQS
8,9,Programming,Rq5gJVxz55Q,Python Full Course for Beginners (13 Hours) – ...,0.366625,0.709210,48,3.891820,0.437685,0.160467,0.398799,0.559266,IQS
9,10,Data Science,hDKCxebp88A,Machine Learning with Python and Scikit-Learn ...,0.241665,0.798634,64,4.174387,0.455004,0.109959,0.435252,0.545211,IQS



DOMAIN-WISE HYBRID SUMMARY


...


TOP 10 VIDEOS MOVING UP FROM FES RANK


,hybrid_rank,fes_rank,movement_from_fes,domain,video_id,video_title,final_engagement_score,intrinsic_quality_score,hybrid_score
15,16,90,74,Data Science,NWONeJKn6kc,Machine Learning Course for Beginners,0.177801,0.753801,0.510228
39,40,98,58,Mathematics,QCPJ0VdpM00,Linear Algebra for Machine Learning,0.142050,0.743015,0.473692
9,10,65,55,Data Science,hDKCxebp88A,Machine Learning with Python and Scikit-Learn ...,0.241665,0.798634,0.545211
38,39,87,48,Mathematics,rSjt1E9WHaQ,Linear Algebra Course – Mathematics for Machin...,0.190916,0.703371,0.475962
26,27,75,48,Programming,q3AuP01daL4,Python Full Course for Beginners,0.224189,0.700071,0.489443
36,37,84,47,Mathematics,JnTa9XtvmfI,Linear Algebra - Full College Course,0.200263,0.702927,0.478736
20,21,66,45,Programming,H2EJuAcrZYU,Python Full Course for Beginners | Complete Al...,0.238742,0.701686,0.494709
11,12,55,43,Programming,8DvywoWv6fI,Python for Everybody - Full University Python ...,0.258595,0.723344,0.525277
53,54,89,35,Physics,AufmV0P6mA0,"001 Introduction to Quantum Mechanics, Probabi...",0.181035,0.650061,0.444177
64,65,97,32,Programming,eWRfhZUzrAc,Python for Beginners – Full Course [Programmin...,0.143859,0.643173,0.418378



TOP 10 VIDEOS MOVING DOWN FROM FES RANK


,hybrid_rank,fes_rank,movement_from_fes,domain,video_id,video_title,final_engagement_score,intrinsic_quality_score,hybrid_score
88,89,30,-59,Mathematics,osEADxaIKIc,Gilbert Strang: Linear Algebra vs Calculus,0.313442,0.394151,0.357815
92,93,35,-58,Mathematics,2RDrcR7urdA,The Best Way To Learn Linear Algebra,0.305599,0.372846,0.342310
86,87,33,-54,Physics,ti1SA02ZiTA,Quantum Physics for Dummies (A Quick Crash Cou...,0.306178,0.407936,0.362333
55,56,14,-42,Data Science,adhgNQmPXZU,Machine Learning (தமிழில்) in 25 Minutes: Abs...,0.355633,0.510049,0.438497
84,85,47,-38,Physics,fcfQkxwz4Oo,Brian Cox explains quantum mechanics in 60 sec...,0.282965,0.427636,0.370558
76,77,41,-36,Data Science,fXojHUuBnaY,I Tried 50 Machine Learning Courses: Here are ...,0.297037,0.458381,0.389882
89,90,56,-34,Biology,pmjiHzNi4A0,How to study and pass Anatomy & Physiology!,0.257248,0.433734,0.354103
91,92,59,-33,Data Science,7gEzEAIELgQ,Machine Learning Explained🔥in 30 Seconds. #sho...,0.246084,0.428161,0.344369
51,52,20,-32,Data Science,sbmPMtQkjxY,Machine Learning Full Course #Education #Ai #M...,0.343485,0.475429,0.446787
72,73,44,-29,Biology,Ae4MadKPJC0,Human Body 101 | National Geographic,0.292943,0.474903,0.396515



PHASE 9F — FINAL VALIDATION
✅ 99 hybrid records
✅ 99 unique video IDs
✅ Alpha values within [0,1]
✅ IQS weights within [0,1]
✅ All Hybrid scores finite
✅ No missing Hybrid scores
✅ Hybrid ranking is complete 1–99

Maximum contribution error: 0.000000000000
✅ Hybrid = FES contribution + IQS contribution
Maximum alpha error: 0.000000000000
✅ Alpha formula validated

✅ PHASE 9F VALIDATION PASSED
Adaptive Hybrid Ranking successfully computed for all 99 videos.

Full Hybrid file:
/content/revision_phase9f/phase9f_hybrid_full.csv

Top-10 file:
/content/revision_phase9f/phase9f_hybrid_top10.csv

Top-20 file:
/content/revision_phase9f/phase9f_hybrid_top20.csv

Domain summary:
/content/revision_phase9f/phase9f_domain_summary.csv

Alpha summary:
/content/revision_phase9f/phase9f_alpha_summary.csv


In [50]:
# ============================================================
# PHASE 9G-A — INDEPENDENT HUMAN EVALUATION DATASET
# ============================================================
#
# PURPOSE:
#   Create a blinded annotation sheet for independent
#   human ground-truth evaluation.
#
# IMPORTANT:
#   FES / IQS / Hybrid scores are NOT included.
#   Human labels must be independent of our system scores.
#
# Relevance:
#   0 = Not relevant
#   1 = Marginally relevant
#   2 = Clearly relevant
#   3 = Highly relevant
#
# Educational Quality:
#   1 = Very poor
#   2 = Poor
#   3 = Adequate
#   4 = Good
#   5 = Excellent
#
# ============================================================

import os
import uuid
import hashlib
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

FES_PATH = (
    "/content/revision_phase9d/"
    "phase9d_fes_full.csv"
)

OUTPUT_ROOT = (
    "/content/revision_phase9g"
)

os.makedirs(
    OUTPUT_ROOT,
    exist_ok=True
)


# ------------------------------------------------------------
# 2. LOAD VIDEO METADATA
# ------------------------------------------------------------

fes_df = pd.read_csv(
    FES_PATH
)

print(
    f"Loaded records: {len(fes_df)}"
)

print(
    f"Unique videos: "
    f"{fes_df['video_id'].nunique()}"
)


# ------------------------------------------------------------
# 3. REQUIRED COLUMNS
# ------------------------------------------------------------

required_columns = [
    "video_id",
    "domain",
    "query",
    "video_title",
    "description"
]

missing_columns = [
    c
    for c in required_columns
    if c not in fes_df.columns
]

if missing_columns:

    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )


# ------------------------------------------------------------
# 4. VALIDATE COHORT
# ------------------------------------------------------------

if len(fes_df) != 99:

    raise ValueError(
        f"Expected 99 videos, found {len(fes_df)}"
    )

if fes_df[
    "video_id"
].nunique() != 99:

    raise ValueError(
        "Video IDs are not unique."
    )

print(
    "✅ 99-video evaluation cohort validated."
)


# ------------------------------------------------------------
# 5. BUILD BLINDED DATASET
# ------------------------------------------------------------
#
# We intentionally exclude:
#
#   final_engagement_score
#   intrinsic_quality_score
#   hybrid_score
#   alpha
#   automated ranking
#
# ------------------------------------------------------------

annotation_df = fes_df[
    [
        "video_id",
        "domain",
        "query",
        "video_title",
        "description"
    ]
].copy()


# ------------------------------------------------------------
# 6. CREATE ANONYMOUS EVALUATION ID
# ------------------------------------------------------------
#
# The evaluator works with an anonymous ID rather than
# the actual YouTube video ID.
#
# This reduces direct association with our system.
# ------------------------------------------------------------

def make_eval_id(video_id):

    digest = hashlib.sha256(
        str(video_id).encode(
            "utf-8"
        )
    ).hexdigest()

    return (
        "EV"
        + digest[:10].upper()
    )


annotation_df[
    "evaluation_id"
] = (
    annotation_df[
        "video_id"
    ]
    .apply(make_eval_id)
)


# ------------------------------------------------------------
# 7. SHUFFLE EVALUATION ORDER
# ------------------------------------------------------------

RANDOM_SEED = 2026

annotation_df = annotation_df.sample(
    frac=1,
    random_state=RANDOM_SEED
).reset_index(
    drop=True
)

annotation_df[
    "evaluation_order"
] = np.arange(
    1,
    len(annotation_df) + 1
)


# ------------------------------------------------------------
# 8. ADD HUMAN-ANNOTATION COLUMNS
# ------------------------------------------------------------

annotation_df[
    "relevance_score"
] = ""

annotation_df[
    "educational_quality_score"
] = ""

annotation_df[
    "justification"
] = ""


# ------------------------------------------------------------
# 9. ADD ANNOTATOR INFORMATION
# ------------------------------------------------------------

annotation_df[
    "annotator_id"
] = ""

annotation_df[
    "evaluation_date"
] = ""


# ------------------------------------------------------------
# 10. COLUMN ORDER
# ------------------------------------------------------------

annotation_df = annotation_df[
    [
        "evaluation_order",
        "evaluation_id",
        "domain",
        "query",
        "video_title",
        "description",
        "relevance_score",
        "educational_quality_score",
        "justification",
        "annotator_id",
        "evaluation_date"
    ]
]


# ------------------------------------------------------------
# 11. SAVE BLINDED ANNOTATION SHEET
# ------------------------------------------------------------

annotation_path = os.path.join(
    OUTPUT_ROOT,
    "phase9g_human_annotation_sheet.csv"
)

annotation_df.to_csv(
    annotation_path,
    index=False
)


# ------------------------------------------------------------
# 12. SAVE CODEBOOK
# ------------------------------------------------------------

codebook = pd.DataFrame({
    "field": [
        "relevance_score",
        "educational_quality_score",
        "justification"
    ],

    "allowed_values": [
        "0, 1, 2, 3",
        "1, 2, 3, 4, 5",
        "Free text"
    ],

    "meaning": [
        (
            "0=Not relevant; "
            "1=Marginally relevant; "
            "2=Clearly relevant; "
            "3=Highly relevant"
        ),
        (
            "1=Very poor; "
            "2=Poor; "
            "3=Adequate; "
            "4=Good; "
            "5=Excellent"
        ),
        (
            "Brief reason for the assigned scores"
        )
    ]
})

codebook_path = os.path.join(
    OUTPUT_ROOT,
    "phase9g_annotation_codebook.csv"
)

codebook.to_csv(
    codebook_path,
    index=False
)


# ------------------------------------------------------------
# 13. DOMAIN DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("PHASE 9G-A — EVALUATION COHORT")
print("=" * 75)

display(
    annotation_df[
        "domain"
    ]
    .value_counts()
    .sort_index()
    .to_frame(
        "videos"
    )
)


# ------------------------------------------------------------
# 14. VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("PHASE 9G-A — VALIDATION")
print("=" * 75)

validation_passed = True


if len(annotation_df) != 99:

    validation_passed = False

    print(
        "❌ Expected 99 videos."
    )

else:

    print(
        "✅ 99 videos present."
    )


if annotation_df[
    "evaluation_id"
].nunique() != 99:

    validation_passed = False

    print(
        "❌ Duplicate evaluation IDs."
    )

else:

    print(
        "✅ 99 unique evaluation IDs."
    )


# Ensure system scores are not present
for forbidden_column in [
    "final_engagement_score",
    "intrinsic_quality_score",
    "hybrid_score",
    "alpha",
    "fes_rank",
    "iqs_rank",
    "hybrid_rank"
]:

    if forbidden_column in annotation_df.columns:

        validation_passed = False

        print(
            f"❌ System score leaked: "
            f"{forbidden_column}"
        )


if validation_passed:

    print(
        "✅ No automated ranking scores "
        "are present in the annotation sheet."
    )


print("\n" + "=" * 75)

if validation_passed:

    print(
        "✅ PHASE 9G-A VALIDATION PASSED"
    )

else:

    print(
        "⚠️ PHASE 9G-A REQUIRES REVIEW"
    )

print("=" * 75)


print(
    f"\nAnnotation sheet:\n"
    f"{annotation_path}"
)

print(
    f"\nCodebook:\n"
    f"{codebook_path}"
)

Loaded records: 99
Unique videos: 99
✅ 99-video evaluation cohort validated.

PHASE 9G-A — EVALUATION COHORT


,videos
domain,
Biology,20
Data Science,20
Mathematics,20
Physics,19
Programming,20



PHASE 9G-A — VALIDATION
✅ 99 videos present.
✅ 99 unique evaluation IDs.
✅ No automated ranking scores are present in the annotation sheet.

✅ PHASE 9G-A VALIDATION PASSED

Annotation sheet:
/content/revision_phase9g/phase9g_human_annotation_sheet.csv

Codebook:
/content/revision_phase9g/phase9g_annotation_codebook.csv


In [51]:
# ============================================================
# PHASE 9G-B — INDEPENDENT ANNOTATOR COPIES
# ============================================================

import os
import pandas as pd

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

INPUT_PATH = (
    "/content/revision_phase9g/"
    "phase9g_human_annotation_sheet.csv"
)

OUTPUT_ROOT = "/content/revision_phase9g"

ANNOTATOR1_PATH = os.path.join(
    OUTPUT_ROOT,
    "phase9g_annotator1.csv"
)

ANNOTATOR2_PATH = os.path.join(
    OUTPUT_ROOT,
    "phase9g_annotator2.csv"
)


# ------------------------------------------------------------
# 2. LOAD MASTER SHEET
# ------------------------------------------------------------

master_df = pd.read_csv(
    INPUT_PATH
)

print(
    f"Master records: {len(master_df)}"
)

print(
    f"Unique evaluation IDs: "
    f"{master_df['evaluation_id'].nunique()}"
)


# ------------------------------------------------------------
# 3. CREATE TWO INDEPENDENT COPIES
# ------------------------------------------------------------

annotator1_df = master_df.copy()
annotator2_df = master_df.copy()


# ------------------------------------------------------------
# 4. CLEAR ANNOTATION FIELDS
# ------------------------------------------------------------

annotation_columns = [
    "relevance_score",
    "educational_quality_score",
    "justification",
    "annotator_id",
    "evaluation_date"
]

for column in annotation_columns:

    annotator1_df[column] = ""
    annotator2_df[column] = ""


# ------------------------------------------------------------
# 5. SAVE
# ------------------------------------------------------------

annotator1_df.to_csv(
    ANNOTATOR1_PATH,
    index=False
)

annotator2_df.to_csv(
    ANNOTATOR2_PATH,
    index=False
)


# ------------------------------------------------------------
# 6. VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("PHASE 9G-B — VALIDATION")
print("=" * 75)

for name, df in [
    ("Annotator 1", annotator1_df),
    ("Annotator 2", annotator2_df)
]:

    print(
        f"\n{name}:"
    )

    print(
        f"  Videos       : {len(df)}"
    )

    print(
        f"  Unique IDs   : "
        f"{df['evaluation_id'].nunique()}"
    )

    print(
        f"  Blank labels : "
        f"{df['relevance_score'].isna().sum()}"
    )

    # Ensure automated scores are absent
    forbidden = [
        "final_engagement_score",
        "intrinsic_quality_score",
        "hybrid_score",
        "alpha",
        "fes_rank",
        "iqs_rank",
        "hybrid_rank"
    ]

    leaked = [
        c for c in forbidden
        if c in df.columns
    ]

    if leaked:

        raise ValueError(
            f"Automated score leakage in {name}: {leaked}"
        )

    print(
        "  Score leakage: none"
    )


# Ensure both sheets contain exactly the same evaluation IDs

ids1 = set(
    annotator1_df["evaluation_id"]
)

ids2 = set(
    annotator2_df["evaluation_id"]
)

if ids1 != ids2:

    raise ValueError(
        "Annotator cohorts do not match."
    )


print("\n" + "=" * 75)
print(
    "✅ PHASE 9G-B VALIDATION PASSED"
)
print("=" * 75)

print(
    f"\nAnnotator 1:\n{ANNOTATOR1_PATH}"
)

print(
    f"\nAnnotator 2:\n{ANNOTATOR2_PATH}"
)

Master records: 99
Unique evaluation IDs: 99

PHASE 9G-B — VALIDATION

Annotator 1:
  Videos       : 99
  Unique IDs   : 99
  Blank labels : 0
  Score leakage: none

Annotator 2:
  Videos       : 99
  Unique IDs   : 99
  Blank labels : 0
  Score leakage: none

✅ PHASE 9G-B VALIDATION PASSED

Annotator 1:
/content/revision_phase9g/phase9g_annotator1.csv

Annotator 2:
/content/revision_phase9g/phase9g_annotator2.csv


validation

PHASE 1 — Data Collection Validation

Coverage

In [ ]:
print("Unique videos:", videos_df["video_id"].nunique())
print("Unique comments:", comments_df["comment_id"].nunique())

Missing Data Check

In [ ]:
print(videos_df.isnull().sum())
print(comments_df.isnull().sum())

Comment Distribution Plot

In [ ]:
import matplotlib.pyplot as plt

comments_df.groupby("video_id").size().plot(kind="bar")
plt.title("Comments per Video")
plt.show()

PHASE 2 — Cleaning Validation

Before vs After Length

In [ ]:
comments_df["original_len"] = comments_df["comment_text"].astype(str).apply(len)
comments_df["clean_comment"] = comments_df["clean_comment"].fillna("").astype(str)

comments_df["original_len"] = comments_df["comment_text"].fillna("").astype(str).apply(len)
comments_df["clean_len"] = comments_df["clean_comment"].apply(len)

print(comments_df[["original_len","clean_len"]].describe())

#print(comments_df[["original_len","clean_len"]].describe())

In [ ]:
print("Empty cleaned comments:",
      (comments_df["clean_comment"].str.strip() == "").sum())

In [ ]:
import matplotlib.pyplot as plt

plt.hist(comments_df["clean_len"], bins=30)
plt.title("Cleaned Comment Length Distribution")
plt.xlabel("Length")
plt.ylabel("Frequency")
plt.show()

PHASE 3 — Semantic Relevance Validation

Check Range

In [ ]:
print(comments_df["relevance_score"].describe())

In [ ]:
plt.hist(comments_df["relevance_score"], bins=30)
plt.title("Relevance Score Distribution")
plt.show()

PHASE 4 — Sentiment Validation

In [ ]:
print(relevant_df["sentiment_score"].describe())

In [ ]:
plt.hist(relevant_df["sentiment_score"], bins=30)
plt.title("Sentiment Score Distribution")
plt.show()

PHASE 5 — Emotional Engagement Validation

Check Bounds

In [ ]:
print(relevant_df["emotion_weighted_score"].describe())

In [ ]:
plt.hist(relevant_df["emotion_weighted_score"], bins=30)
plt.title("emotion_weighted_score Distribution")
plt.show()

emotional_variance

In [ ]:
print(video_emotion_df["emotional_variance"].describe())

PHASE 6 — Engagement Score Validation

check range

In [ ]:
print("Available columns:", df.columns)


In [ ]:
print(df["final_engagement_score_x"].describe())

Correlation with Likes

In [ ]:
from scipy.stats import spearmanr
phase6_full = pd.read_csv("phase6_full.csv")
from scipy.stats import spearmanr

print(spearmanr(
    phase6_full["likes"],
    phase6_full["final_engagement_score"]
))

Correlation with comment count

In [ ]:
print(spearmanr(
    phase6_full["relevant_comment_count"],
    phase6_full["final_engagement_score"]
))

Correlation with emotional engagement

In [ ]:
print(spearmanr(
    phase6_full["emotional_engagement"],
    phase6_full["final_engagement_score"]
))

In [ ]:
from scipy.stats import spearmanr

metrics = [
    "likes",
    "relevant_comment_count",
    "emotional_engagement",
    "emotional_variance"
]

for m in metrics:
    corr = spearmanr(phase6_full[m], phase6_full["final_engagement_score"])
    print(f"{m} vs ES:", corr)

PHASE 7 — ITQS Validation

In [ ]:
print(videos_df["ITQS"].describe())

In [ ]:
print(spearmanr(videos_df["semantic_alignment"], videos_df["ITQS"]))
print(spearmanr(videos_df["teaching_score"], videos_df["ITQS"]))

PHASE 8 — Hybrid Validation

check bounds

In [ ]:
print(df["hybrid_score"].describe())

In [ ]:
import seaborn as sns

corr = df[["final_engagement_score_x",
           "ITQS_x",
           "hybrid_score"]].corr(method="spearman")
sns.heatmap(corr, annot=True, cmap="coolwarm")
plt.title("Spearman Correlation Matrix")
plt.show()

Top-K Overlap Analysis

1️⃣ Rank Stability (Bootstrap)

Resample comments → recompute ES → recompute hybrid → measure rank change.

That proves robustness.

2️⃣ Sensitivity Analysis of k

Plot α as function of comment count.

In [ ]:
top_es = set(df.sort_values("final_engagement_score_x", ascending=False).head(5)["video_id"])
top_itqs = set(df.sort_values("ITQS_x", ascending=False).head(5)["video_id"])
top_hybrid = set(df.sort_values("hybrid_score", ascending=False).head(5)["video_id"])

print("ES-Hybrid overlap:", len(top_es & top_hybrid))
print("ITQS-Hybrid overlap:", len(top_itqs & top_hybrid))

testing

In [ ]:
df_analysis = df_clean.copy()

df_analysis["rank_ES"] = df_analysis["final_engagement_score"].rank(ascending=False, method="dense")
df_analysis["rank_ITQS"] = df_analysis["ITQS"].rank(ascending=False, method="dense")
df_analysis["rank_Hybrid"] = df_analysis["hybrid_score"].rank(ascending=False, method="dense")

df_analysis.head()

In [ ]:
high_itqs = df_analysis.sort_values("rank_ITQS").head(5)
high_itqs

In [ ]:
df_analysis["hybrid_vs_ES_shift"] = df_analysis["rank_ES"] - df_analysis["rank_Hybrid"]
df_analysis["hybrid_vs_ITQS_shift"] = df_analysis["rank_ITQS"] - df_analysis["rank_Hybrid"]

df_analysis.sort_values("hybrid_vs_ES_shift", ascending=False).head()

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(df_analysis["rank_ES"], df_analysis["rank_Hybrid"])
plt.xlabel("Engagement Rank")
plt.ylabel("Hybrid Rank")
plt.title("Hybrid vs Engagement Rank")
plt.show()

In [ ]:
improved = (df_analysis["hybrid_vs_ES_shift"] > 0).sum()
print("Videos improved over ES ranking:", improved)

In [ ]:
df_analysis["hybrid_vs_ES_shift"].mean()

BASE LINE COMPARISION

Likes-Based Ranking (Baseline 1)

In [ ]:
df["rank_likes_baseline"] = df["likes"].rank(
    ascending=False,
    method="dense"
)

df[["video_id","video_title","likes","rank_likes_baseline"]].sort_values("rank_likes_baseline").head()

Comment Count Ranking (Baseline 2)

In [ ]:
df["rank_comments_baseline"] = df["relevant_comment_count_x"].rank(
    ascending=False,
    method="dense"
)

df[["video_title","relevant_comment_count_x","rank_comments_baseline"]].sort_values("rank_comments_baseline").head()

Views Ranking (Recommended Baseline)

In [ ]:
df["rank_views_baseline"] = df["views"].rank(
    ascending=False,
    method="dense"
)

df[["video_title","views","rank_views_baseline"]].sort_values("rank_views_baseline").head()

Compare Baselines with Hybrid Model

In [ ]:
cols = [
    "rank_views_baseline",
    "rank_likes_baseline",
    "rank_comments_baseline",
    "final_rank"
]

df[["video_id","video_title"] + cols].sort_values("final_rank").head(10)

Spearman Comparison with Hybrid

In [ ]:
print(df.columns)

In [ ]:
from scipy.stats import spearmanr

baselines = [
    "rank_views_baseline",
    "rank_likes_baseline",
    "rank_comments_baseline"
]

for b in baselines:
    rho, p = spearmanr(df[b], df["final_rank"])
    print(b, "vs Hybrid →", rho)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))

for i in range(len(df)):

    plt.plot(
        [1,2],
        [df["rank_views_baseline"].iloc[i], df["final_rank"].iloc[i]],
        marker="o",
        alpha=0.6
    )

plt.xticks([1,2], ["Views Rank", "Hybrid Rank"])
plt.ylabel("Rank Position")
plt.title("Ranking Shift: Views Baseline → Hybrid Model")

plt.gca().invert_yaxis()
plt.grid(True)

plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

plt.scatter(df["rank_views_baseline"], df["final_rank"], label="Views", alpha=0.7)
plt.scatter(df["rank_likes_baseline"], df["final_rank"], label="Likes", alpha=0.7)
plt.scatter(df["rank_comments_baseline"], df["final_rank"], label="Comments", alpha=0.7)

plt.xlabel("Baseline Rank")
plt.ylabel("Hybrid Rank")
plt.title("Baseline Rankings vs Hybrid Ranking")

plt.legend()
plt.grid(True)
plt.gca().invert_yaxis()

plt.show()

Ablation Study Code

In [ ]:
from scipy.stats import spearmanr

print("ABLATION STUDY\n")

rho, p = spearmanr(df["rank_views_baseline"], df["final_rank"])
print("Views vs Hybrid")
print("Spearman:", rho)
print("p-value:", p)
print()

rho, p = spearmanr(df["rank_likes_baseline"], df["final_rank"])
print("Likes vs Hybrid")
print("Spearman:", rho)
print("p-value:", p)
print()

rho, p = spearmanr(df["rank_comments_baseline"], df["final_rank"])
print("Comments vs Hybrid")
print("Spearman:", rho)
print("p-value:", p)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

ablation_matrix = df[
    ["final_engagement_score_x", "ITQS_x", "hybrid_score"]
].corr(method="spearman")

sns.heatmap(ablation_matrix, annot=True, cmap="coolwarm")
plt.title("Ablation Study: Model Component Correlation")
plt.show()

Sensitivity Analysis **Code**

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

k_values = [1, 2, 3, 5, 10]

results = []

for k in k_values:

    alpha_k = np.log1p(df["relevant_comment_count_x"]) / (
        np.log1p(df["relevant_comment_count_x"]) + k
    )

    hybrid_k = alpha_k * df["final_engagement_score_x"] + \
               (1 - alpha_k) * df["ITQS_x"]

    rho, p = spearmanr(df["hybrid_score"], hybrid_k)

    results.append({
        "k": k,
        "Spearman_with_original": rho,
        "p_value": p
    })

sens_df = pd.DataFrame(results)

sens_df

In [ ]:
import matplotlib.pyplot as plt

plt.plot(sens_df["k"], sens_df["Spearman_with_original"], marker="o")

plt.xlabel("k value")
plt.ylabel("Spearman Correlation with Original Hybrid")
plt.title("Sensitivity Analysis of k")
plt.grid(True)

plt.show()